# Workbook 2 — Statistical learning and regularization: why the hacky bullshit works

**Solutions · Mysteries of Deep Learning, Iliad Intensive 2026**

Deep learning practice is full of tricks that look like hacks: weight decay,
early stopping, dropout, "flat minima", training straight through the point
where the training loss hits zero. This workbook derives, in models simple
enough to solve exactly, what each trick does and what it does not. The aim is
that when you meet a claim about generalization, you can say which classical
statement it borrows, and in which regime that statement is a theorem. You do
the derivations. Where a result is quoted rather than derived, the text says so,
and each section links the papers or textbook sections its derivations come from.

**Worked solutions.** Every problem has a worked solution in the cell directly
after it in
[`02_statistical_learning_solutions.ipynb`](02_statistical_learning_solutions.ipynb).
[`02_statistical_learning_exercises.ipynb`](02_statistical_learning_exercises.ipynb)
is the same notebook with those cells left blank for your own work. Both are
for everyone. Hints are folded under "Hint".

**Sections.** Choose whatever interests you, in any order. Each section opens
with its result, so you can tell quickly whether you already know it. The
"Uses" column lists what a section actually relies on. The notation block
below covers the shared definitions. §7 and §8 restate what they need; §9's
exercise points back to §4, §7 and §8.

| Section | What you derive or read | Uses |
|---|---|---|
| [1. Risks and gaps](#wb2-section-1) | The generalization gap versus the excess risk; both computed exactly for least squares | — |
| [2. Doesn't some theorem make this impossible? (天无绝人之路)](#wb2-section-2) | No free lunch and an Occam bound; ReLU unit counts and a depth separation; why class-wide capacity bounds are vacuous for real networks. Reading: Zhang et al. (*core*) | §1.A for 2.C; §4 for 2.D.i.2 |
| [3. Bias–variance](#wb2-section-3) | The pointwise decomposition; ridge direction by direction; effective degrees of freedom | §1.B |
| [4. Regularization three ways, and how not to think about it](#wb2-section-4) | Ridge as prior, constraint and spectral filter; early stopping; dropout (optional); when flatness can matter and why raw sharpness cannot. Reading: Dinh et al. (*core*) | §3.B (setup restated; 4.A.i.2 uses 3.B.i.1); §1.B for one check; §1.A.i.3 and §2.C–2.D for 4.B.i.3 |
| [5. Overfitting and double descent](#wb2-section-5) | The least-squares double-descent curve, and why optimal ridge removes the peak | §1.B.ii, §4.A.i.1, §4.C.i.4; §3.A for 5.D.i |
| [6. Information bottleneck and the information plane](#wb2-section-6) | Why mutual information is degenerate for deterministic networks; what binned estimators measure; compression as clustering in a noisy model; the IB as rate–distortion (optional) | — |
| [7. Bayes and MLE generalization error](#wb2-section-7) | The sandwich covariance, TIC, AIC and BIC; optionally, a single fit versus the Bayes ensemble versus a posterior draw | §4.E.i.1 (restated); §1.B for one check; §3–§4 for 7.E.ii |
| [8. Does it cancel? $C$ vs $H$](#wb2-section-8) | Gradient noise versus curvature; the diagonal power law; SGD's stationary covariance versus a posterior | §4.E.i.1, §7.A–7.C (restated) |
| [9. Fantastic generalization measures](#wb2-section-9) | A reading on which published measures predict the gap, with one exercise linking it to §4, §7 and §8 | §4.E.i.3–4, §7 (setup and 7.B), §8.A |

**Notation, used throughout.**

- Data $z=(x,y)\sim P$, and an i.i.d. sample $S=(z_1,\dots,z_n)$.
- A model with parameters $\theta\in\mathbb R^p$ and a loss $\ell(\theta;z)$.
- Population risk $R(\theta)=\mathbb E_{z\sim P}\,\ell(\theta;z)$; empirical risk $\hat R_n(\theta)=\frac1n\sum_{i=1}^n\ell(\theta;z_i)$.
- The empirical risk minimizer (ERM) $\hat\theta_n\in\arg\min_\theta\hat R_n(\theta)$, and the population minimizer $\theta^*\in\arg\min_\theta R(\theta)$. $\theta^*$ is the best the model class can do, not necessarily the truth.
- $n$ is the number of samples and $p$ the number of parameters; §5 uses $\gamma=p/n$.
- The running example is the linear model $y=x^\top\beta+\varepsilon$ with $\mathbb E\varepsilon=0$ and $\mathrm{Var}\,\varepsilon=\sigma^2$. The design matrix is $X\in\mathbb R^{n\times p}$, with rows $x_i^\top$, and $y\in\mathbb R^n$.

<a id="wb2-section-1"></a>

## 1. Risks and gaps

**The result.** The *generalization gap* $R(\hat\theta_n)-\hat R_n(\hat\theta_n)$
and the *excess risk* $R(\hat\theta_n)-R(\theta^*)$ are different quantities.
Either can be large while the other is zero. A held-out set estimates population
risk and hence the gap; excess risk additionally needs the optimum risk. For ERM, the expected gap bounds the expected excess risk. In ordinary
least squares both are exact: the excess risk is $\sigma^2p/(n-p-1)$, about
$\sigma^2/n$ per parameter, and the gap is about twice that.

### 1.A. Two quantities called "generalization"

For any learning rule $S\mapsto\hat\theta_n$, not only ERM, define

$$
\mathrm{gap}_n=R(\hat\theta_n)-\hat R_n(\hat\theta_n),
\qquad
\mathcal E_n=R(\hat\theta_n)-R(\theta^*).
$$

Both are random through $S$. Papers say "generalizes well" for either one,
and sometimes for both in the same paragraph.

#### 1.A.i. Gap without excess, excess without gap

1. Construct a model class and a distribution in which ERM has excess risk exactly zero but a gap equal to the noise variance.
2. Construct a learning rule with expected gap exactly zero but excess risk bounded away from zero.
3. Show that for ERM, $\mathbb E\,\mathcal E_n\le\mathbb E\,\mathrm{gap}_n$, and that $\mathcal E_n\le 2\sup_\theta|R(\theta)-\hat R_n(\theta)|$.
4. You have a held-out set of $m$ fresh samples from $P$. Which of the two quantities can you estimate, and how accurately? What would you need in order to estimate the other?

<details><summary>Hint</summary>

For 1, use squared loss with a continuous input, and let the class contain functions that agree with the best predictor except on a finite set of points. For 2, the rule need not look at the data. For 3, telescope $\mathcal E_n$ through $\hat R_n(\hat\theta_n)$ and $\hat R_n(\theta^*)$, and ask which terms depend on $S$ in a way that matters.
</details>

**1.** Take $x\sim\mathrm{Unif}[0,1]$ and $y=f^*(x)+\varepsilon$ with $\varepsilon\sim\mathcal N(0,\sigma^2)$, under squared loss. Let the class be $f^*$ together with every "patched" function $f_T$, where $T$ is a finite set of labelled points with distinct inputs: $f_T(x)=y$ if $(x,y)\in T$ and $f_T(x)=f^*(x)$ otherwise. Each $f_T$ differs from $f^*$ on a set of measure zero, so $R(f_T)=R(f^*)=\sigma^2$ and every member of the class is a population minimizer. ERM returns $f_S$, which has zero training loss because the $x_i$ are almost surely distinct. Hence

$$
\mathcal E_n=0,\qquad \mathrm{gap}_n=\sigma^2 .
$$

Memorizing costs nothing here because the memorized inputs never recur. This is a caricature of benign overfitting; §5 has a real version.

**2.** Take the rule that ignores the data and outputs a fixed $\theta_0$ with $R(\theta_0)>R(\theta^*)$. Since $\theta_0$ does not depend on $S$, $\mathbb E\hat R_n(\theta_0)=R(\theta_0)$, so $\mathbb E\,\mathrm{gap}_n=0$. Meanwhile $\mathcal E_n=R(\theta_0)-R(\theta^*)>0$ for every $S$. A small gap means the training loss is an honest estimate of the test loss, not that the model is good.

**3.** Write

$$
\mathcal E_n=\underbrace{R(\hat\theta_n)-\hat R_n(\hat\theta_n)}_{\mathrm{gap}_n}
+\underbrace{\hat R_n(\hat\theta_n)-\hat R_n(\theta^*)}_{\le 0\ \text{(ERM)}}
+\underbrace{\hat R_n(\theta^*)-R(\theta^*)}_{\text{mean }0}.
$$

The middle term is non-positive because $\hat\theta_n$ minimizes $\hat R_n$. The last has mean zero because $\theta^*$ does not depend on $S$. Taking expectations gives $\mathbb E\,\mathcal E_n\le\mathbb E\,\mathrm{gap}_n$. Bounding the first and third terms by $\sup_\theta|R-\hat R_n|$ and dropping the middle one gives $\mathcal E_n\le2\sup_\theta|R(\theta)-\hat R_n(\theta)|$. This is the route every capacity bound takes (§2.C): control the worst gap over the whole class, and the excess risk follows.

**4.** Because $\hat\theta_n$ is independent of the held-out data, the held-out mean loss is unbiased for $R(\hat\theta_n)$, with standard error $\mathrm{sd}(\ell)/\sqrt m$. Subtracting the training loss estimates the gap. The excess risk also needs $R(\theta^*)$, the best achievable loss in the class, which a held-out loss alone does not identify without further assumptions or knowledge. You can measure *differences* of excess risk between two fitted predictors in the same model class, since $R(\theta^*)$ cancels. Different classes can have different optimum risks. More generally, a held-out set estimates the population loss of whatever predictor you evaluate on it, whether one network or an ensemble.

A sanity example with both quantities nonzero: for constant predictors on $y\sim\mathcal N(\mu,\sigma^2)$, $\hat\theta_n=\bar y$, $\mathbb E\,\mathcal E_n=\sigma^2/n$ and $\mathbb E\,\mathrm{gap}_n=2\sigma^2/n$.

### 1.B. Ordinary least squares, exactly

Take $y=X\beta+\varepsilon$ with $\varepsilon\sim\mathcal N(0,\sigma^2I_n)$,
$n>p$, and $X$ of full column rank. Use squared loss, so
$\hat R_n(b)=\frac1n\|y-Xb\|^2$. For the population, take a fresh
$x\sim\mathcal N(0,I_p)$ with $y=x^\top\beta+\varepsilon$.

*Source:* least squares and the optimism of the training error are in
[*The Elements of Statistical Learning*](https://hastie.su.domains/ElemStatLearn/)
(Hastie, Tibshirani and Friedman), §3.2 and §7.4.

#### 1.B.i. Mean, covariance and excess risk at a fixed design

1. Derive $\hat\beta=(X^\top X)^{-1}X^\top y$, and show $\mathbb E\hat\beta=\beta$ and $\mathrm{Cov}\,\hat\beta=\sigma^2(X^\top X)^{-1}$. Here expectations are over the noise, with $X$ fixed.
2. Show that $R(b)=\sigma^2+\|b-\beta\|^2$. Hence $\theta^*=\beta$, $\mathcal E_n=\|\hat\beta-\beta\|^2$, and $\mathbb E[\mathcal E_n\mid X]=\sigma^2\,\mathrm{Tr}\,(X^\top X)^{-1}$.
3. Show that the expected training risk is $\mathbb E[\hat R_n(\hat\beta)\mid X]=\sigma^2(n-p)/n$.

<details><summary>Hint</summary>

For 3, the residual is $(I-P_X)\varepsilon$, where $P_X=X(X^\top X)^{-1}X^\top$ is a rank-$p$ orthogonal projection.
</details>

**1.** $\nabla_b\hat R_n=-\frac2nX^\top(y-Xb)=0$ gives the normal equations $X^\top Xb=X^\top y$. Since $X^\top X$ is invertible, $\hat\beta=(X^\top X)^{-1}X^\top y$. Substituting $y=X\beta+\varepsilon$,

$$
\hat\beta=\beta+(X^\top X)^{-1}X^\top\varepsilon,
$$

so $\mathbb E\hat\beta=\beta$, and

$$
\mathrm{Cov}\,\hat\beta=(X^\top X)^{-1}X^\top(\sigma^2I)X(X^\top X)^{-1}=\sigma^2(X^\top X)^{-1}.
$$

**2.** $R(b)=\mathbb E(x^\top\beta+\varepsilon-x^\top b)^2=\sigma^2+(\beta-b)^\top\mathbb E[xx^\top]\,(\beta-b)=\sigma^2+\|b-\beta\|^2$, since $\varepsilon$ is independent of $x$ and $\mathbb E xx^\top=I$. So $\theta^*=\beta$, $R(\theta^*)=\sigma^2$ and $\mathcal E_n=\|\hat\beta-\beta\|^2$. Then

$$
\mathbb E\|\hat\beta-\beta\|^2=\mathrm{Tr}\,\mathrm{Cov}\,\hat\beta=\sigma^2\,\mathrm{Tr}\,(X^\top X)^{-1}.
$$

**3.** The residual is $y-X\hat\beta=(I-P_X)y=(I-P_X)\varepsilon$, because $(I-P_X)X=0$. So $\mathbb E\|y-X\hat\beta\|^2=\sigma^2\,\mathrm{Tr}(I-P_X)=\sigma^2(n-p)$, and dividing by $n$ gives $\sigma^2(n-p)/n$. The fit absorbs the noise along $p$ directions, so the training loss *underestimates* $\sigma^2$.

#### 1.B.ii. Random design: parameter counting

Now let the rows of $X$ be i.i.d. $\mathcal N(0,I_p)$, so $X^\top X$ is Wishart.

1. Show that $\mathbb E[(X^\top X)^{-1}]=I_p/(n-p-1)$ for $n>p+1$. Conclude that $\mathbb E\,\mathcal E_n=\sigma^2p/(n-p-1)$.
2. Compute the expected gap exactly, and show that for $n\gg p$ the excess risk is $\approx\sigma^2p/n$ while the gap is $\approx2\sigma^2p/n$. Where does the factor of two come from?
3. What happens as $p\to n-1$? Which eigenvalues of $X^\top X$ are responsible?

<details><summary>Hint</summary>

Rotation invariance shows that $\mathbb E(X^\top X)^{-1}$ is a multiple of $I$. For a diagonal entry, the block-inverse (Schur complement) formula gives $[(X^\top X)^{-1}]_{11}=1/\big(x_{\cdot1}^\top(I-P_{-1})x_{\cdot1}\big)$. Here $x_{\cdot1}\in\mathbb R^n$ is the first column of $X$ and $P_{-1}$ projects onto the span of the other $p-1$ columns. What is the law of the denominator? You may use $\mathbb E[1/\chi^2_k]=1/(k-2)$.
</details>

**1.** *Proportional to $I$.* For any orthogonal $O\in\mathbb R^{p\times p}$, $XO$ has the same law as $X$, and $((XO)^\top XO)^{-1}=O^\top(X^\top X)^{-1}O$. So $M=\mathbb E(X^\top X)^{-1}$ satisfies $M=O^\top MO$ for all $O$, which forces $M=cI$.

*The constant.* By the Schur complement, $[(X^\top X)^{-1}]_{11}=1/\big(x_{\cdot1}^\top(I-P_{-1})x_{\cdot1}\big)$. Conditional on the other columns, $I-P_{-1}$ is an orthogonal projection of rank $n-p+1$, and it is independent of $x_{\cdot1}\sim\mathcal N(0,I_n)$. So the denominator is $\chi^2_{n-p+1}$. Next, $\mathbb E[1/\chi^2_k]=\frac{\Gamma(k/2-1)}{2\Gamma(k/2)}=\frac1{k-2}$ for $k>2$, which gives $c=1/(n-p-1)$. Combining with 1.B.i.2,

$$
\mathbb E\,\mathcal E_n=\sigma^2\,\mathrm{Tr}\,\mathbb E(X^\top X)^{-1}=\frac{\sigma^2p}{n-p-1}.
$$

**2.** $\mathbb E R(\hat\beta)=\sigma^2+\sigma^2p/(n-p-1)=\sigma^2\frac{n-1}{n-p-1}$, and by 1.B.i.3, $\mathbb E\hat R_n(\hat\beta)=\sigma^2\frac{n-p}{n}$. So

$$
\mathbb E\,\mathrm{gap}_n=\sigma^2\Big[\frac{n-1}{n-p-1}-\frac{n-p}{n}\Big]
\approx\sigma^2\Big[\Big(1+\frac pn\Big)-\Big(1-\frac pn\Big)\Big]=\frac{2\sigma^2p}{n}.
$$

The test loss *overestimates* $\sigma^2$ by about $\sigma^2p/n$ (the excess risk), and the training loss *underestimates* it by the same amount, because the fit has absorbed the noise along $p$ directions. The gap counts both. This is the factor of two between "optimism" and "excess risk" that §7 derives in general (TIC vs $\tfrac1{2n}\mathrm{Tr}(H^{-1}C)$).

**3.** Each of the $p$ coefficients is estimated with variance $\approx\sigma^2/n$, which is "parameter counting". The exact correction $n/(n-p-1)$ diverges as $p\to n-1$. The cause is the smallest eigenvalues of $X^\top X/n$. When $p/n\to\gamma<1$, its spectrum fills $[(1-\sqrt\gamma)^2,(1+\sqrt\gamma)^2]$ (Marchenko–Pastur, quoted), and $\mathrm{Tr}(X^\top X)^{-1}=\frac1n\sum_i 1/\lambda_i(X^\top X/n)$ is dominated by the eigenvalues near the lower edge, which tends to $0$ as $\gamma\to1$. Noise along nearly degenerate directions of the design is amplified without limit. §5 picks this up.

(All of these formulas are checked by Monte Carlo in `checks/wb2a_check.py`.)

When someone says deep nets "generalize mysteriously", ask which quantity they
mean. Zero training loss with a small test loss is a statement about the gap
(and the test loss). A statement that more parameters do not hurt is about the
excess risk, or about the test loss as a function of $p$.

<a id="wb2-section-2"></a>

## 2. Doesn't some theorem make this impossible? (天无绝人之路)

**The result.** Three classical theorems seem to say that learning from data
should be impossible, or at least unexplained. No free lunch says every learner
is at chance off the training set. Universal approximation says nets can
represent anything, so why this function? The elementary class-wide bounds below are
vacuous at the network sizes considered. Each theorem is true, and none of them bites. NFL
averages over a uniform prior on targets that the world does not have. UAT is
an existence statement, silent on which function is learned or at what cost.
Capacity bounds control the worst function in the class, and Zhang et al.
showed that the class networks actually reach includes memorizers of random
labels. 天无绝人之路, "heaven never closes every road": the way out is a bet on
structure in the targets, made implicitly by the architecture and the training
procedure.

### 2.A. No free lunch

Let the input domain $\mathcal X$ be finite, with $|\mathcal X|=N$, and the
target a binary function $f:\mathcal X\to\{0,1\}$. A training set consists of
inputs $x_1,\dots,x_n$, chosen independently of $f$, with labels $f(x_i)$. Assume at least one input remains unseen. A learner
is any map, possibly randomized, from the labelled sample to a hypothesis
$h:\mathcal X\to\{0,1\}$. With $U$ the set of unseen inputs, the off-training-set
error is
$\mathrm{err}_{\mathrm{OTS}}(h,f)=\frac1{|U|}\sum_{x\in U}\mathbf 1[h(x)\ne f(x)]$.

*Sources:* off-training-set NFL is from
[Wolpert (1996)](https://doi.org/10.1162/neco.1996.8.7.1341). Shalev-Shwartz
and Ben-David's
[*Understanding Machine Learning*](http://www.cs.huji.ac.il/~shais/UnderstandingMachineLearning)
(free PDF; "UML" below) proves a PAC version of NFL in §5.1, the finite-class
bound of part 3 in §2.3.1, and the description-length (Occam) version in §7.3.

#### 2.A.i. Prove NFL, then find the bet

1. Let $f$ be uniform over all $2^N$ functions. Prove that every learner has expected off-training-set error exactly $\tfrac12$, conditional on any training inputs.
2. Conclude that for any two learners $A$ and $B$, $\sum_f\big[\mathrm{err}_{\mathrm{OTS}}(A,f)-\mathrm{err}_{\mathrm{OTS}}(B,f)\big]=0$. What does this say about a learner that beats another on some targets?
3. Now let $f$ lie in a class $\mathcal F$ with $|\mathcal F|=M$, and let the inputs be i.i.d. from a distribution $D$ on $\mathcal X$. Show that a learner returning any $h\in\mathcal F$ consistent with the sample satisfies $\Pr[\mathrm{err}_D(h)>\epsilon]\le Me^{-n\epsilon}$, where $\mathrm{err}_D$ is the error under $D$. How many samples suffice if $\mathcal F$ is the set of functions computed by programs of at most $b$ bits? What does the bound give for the class of all functions?
4. Where in the proof of 1 did the uniform prior enter? Give priors over targets under which the training labels are informative about the unseen ones, and say which kind of learner each favours.

<details><summary>Hint</summary>

For 1, condition on the training labels. Under the uniform prior, the values $f(x)$ are independent fair coins, so what is the conditional law of $f$ on $U$? For 3, a fixed $h$ with error above $\epsilon$ survives $n$ i.i.d. samples with probability at most $(1-\epsilon)^n$. Then take a union bound.
</details>

**1.** Under the uniform prior, the values $\{f(x)\}_{x\in\mathcal X}$ are i.i.d. fair coins. Condition on the training inputs and their labels. The values $\{f(x)\}_{x\in U}$ are still i.i.d. fair coins, independent of everything the learner sees. The hypothesis $h$ is a function of the training labels and of independent internal randomness, so $h$ is independent of $f|_U$. Hence $\Pr[h(x)\ne f(x)]=\tfrac12$ for each $x\in U$, and the average over $U$ has expectation $\tfrac12$.

**2.** By 1, $\sum_f\mathrm{err}_{\mathrm{OTS}}(A,f)=2^N\cdot\tfrac12=\sum_f\mathrm{err}_{\mathrm{OTS}}(B,f)$, so the difference sums to zero. Whatever advantage $A$ has over $B$ on one set of targets, it gives back exactly on the complement. No learner is better than another "in general", even the learner that predicts the opposite of the majority label.

**3.** Fix $h$ with $\mathrm{err}_D(h)>\epsilon$. Each i.i.d. sample is consistent with it with probability below $1-\epsilon$, so $\Pr[h\text{ consistent with all }n]\le(1-\epsilon)^n\le e^{-n\epsilon}$. There are at most $M$ such $h$, so by the union bound $\Pr[\exists\text{ consistent }h\text{ with }\mathrm{err}_D(h)>\epsilon]\le Me^{-n\epsilon}$. Equivalently, with probability at least $1-\delta$, every consistent $h$ has $\mathrm{err}_D(h)\le(\ln M+\ln(1/\delta))/n$. For programs of at most $b$ bits, $M<2^{b+1}$, so $n\approx(b\ln2+\ln(1/\delta))/\epsilon$ samples suffice. The cost of learning scales with the *description length* of the target (Occam). For the class of all functions, $M=2^N$, and the bound needs $n\gtrsim N\ln2/\epsilon$: you must see a constant fraction of the domain, which is NFL again.

**4.** It entered at "$f|_U$ is independent of the labels on the training set". Any prior under which seen labels are informative about unseen ones escapes NFL. Examples:
- a simplicity (Solomonoff) prior, which favours the shortest consistent program (WB4);
- a smoothness prior, which favours kernel or nearest-neighbour methods;
- a symmetry prior, which favours equivariant architectures such as CNNs (WB4).

The question "why does deep learning generalize?" becomes: which prior over targets do the architecture and the optimizer implement, and why does it match the world?

### 2.B. Universal approximation: existence, not mechanism

**UAT (quoted).** For any continuous $f$ on a compact $K\subset\mathbb R^d$ and
any $\epsilon>0$, a one-hidden-layer network with a continuous non-polynomial
activation (for example ReLU) achieves $\sup_K|f-g|<\epsilon$. Cybenko (1989)
and Hornik (1991) proved the sigmoidal versions; the non-polynomial
characterization is
[Leshno, Lin, Pinkus and Schocken (1993)](https://linkinghub.elsevier.com/retrieve/pii/S0893608005801315).
The theorem says nothing about how many units are needed. The problem below
counts them in one dimension. Part 4 follows the depth-separation construction
of [Telgarsky (2015)](https://arxiv.org/abs/1509.08101).

#### 2.B.i. How many units for $\sin\omega x$?

1. Show that a continuous piecewise-linear function on $[0,1]$ with $K$ pieces is exactly a one-hidden-layer ReLU network with $K$ units plus an output bias.
2. *Upper bound.* Show that the piecewise-linear interpolant of $\sin(\omega x)$ at $K+1$ equally spaced knots has sup error at most $\omega^2/(8K^2)$. Deduce that $\lceil\omega/\sqrt{8\epsilon}\,\rceil$ units suffice for accuracy $\epsilon$.
3. *Lower bound.* Show that a one-hidden-layer ReLU network with $m$ units on $\mathbb R$ is piecewise linear with at most $m+1$ pieces. Then show that for $\epsilon<1$ accuracy, $m$ must grow at least like $\omega/(2\pi)$.
4. *Depth.* Let $t(x)=2\,\mathrm{ReLU}(x)-4\,\mathrm{ReLU}(x-\tfrac12)$, the tent map on $[0,1]$. Show that the $k$-fold composition $t^{\circ k}$ has $2^k$ linear pieces. Conclude that a depth-$k$ network with $2k$ units computes a function that any one-hidden-layer network needs at least $2^k-1$ units to represent.
5. List three things UAT does not tell you.

<details><summary>Hint</summary>

For 2, the interpolation error on an interval $[a,b]$ is $e(x)=\tfrac12f''(\xi)(x-a)(x-b)$. For 3, the extrema of $\sin\omega x$ alternate between $+1$ and $-1$. On a piece where $g$ is affine, can $g$ be positive, negative, positive at three consecutive extrema? For 4, $t$ maps each half of $[0,1]$ monotonically onto $[0,1]$.
</details>

**1.** Let the breakpoints be $0=t_0<t_1<\dots<t_{K-1}<1$, with slopes $a_0,\dots,a_{K-1}$ on the pieces. Then on $[0,1]$

$$
g(x)=g(0)+a_0\,\mathrm{ReLU}(x-t_0)+\sum_{k=1}^{K-1}(a_k-a_{k-1})\,\mathrm{ReLU}(x-t_k),
$$

since each ReLU switches on the slope change at its breakpoint. That is $K$ units plus a bias.

**2.** On a knot interval of length $h=1/K$, the interpolation error is $e(x)=\tfrac12f''(\xi)(x-a)(x-b)$, and $|(x-a)(x-b)|\le h^2/4$. With $|f''|\le\omega^2$ this gives $|e|\le\omega^2h^2/8=\omega^2/(8K^2)$. Setting $\omega^2/(8K^2)\le\epsilon$ gives $K\ge\omega/\sqrt{8\epsilon}$, so $m=\lceil\omega/\sqrt{8\epsilon}\rceil$ units suffice. The count is linear in the frequency.

**3.** Each unit $\mathrm{ReLU}(wx+b)$ has at most one breakpoint, so the sum has at most $m$ breakpoints and at most $m+1$ affine pieces. The extrema of $\sin\omega x$ in $[0,1]$ are at $x_j=(\pi/2+j\pi)/\omega$, with values $(-1)^j$, and there are $J=\lfloor\omega/\pi-\tfrac12\rfloor+1$ of them. If $\sup|g-\sin\omega x|<1$, then $g(x_j)$ has sign $(-1)^j$. An affine function that is positive at $x_j$ and $x_{j+2}$ is positive at $x_{j+1}$, which lies between them. So a single piece contains at most two consecutive extrema, and $J\le2(m+1)$, i.e. $m\ge J/2-1\approx\omega/(2\pi)$. Together with 2, the number of units needed is $\Theta(\omega)$ at fixed accuracy.

**4.** On $[0,\tfrac12]$, $t(x)=2x$ is an increasing bijection onto $[0,1]$; on $[\tfrac12,1]$, $t(x)=2-2x$ is a decreasing bijection onto $[0,1]$. So $t^{\circ k}=t^{\circ(k-1)}\circ t$ consists of two reparametrized copies of $t^{\circ(k-1)}$, one of them reversed, and by induction it has $2\cdot2^{k-1}=2^k$ pieces. Each layer uses 2 units, so the depth-$k$ network has $2k$ units. By 3, one hidden layer needs $m+1\ge2^k$ units. Depth buys pieces exponentially.

**5.** UAT does not tell you:
- *how many units* are needed. For generic Lipschitz targets in $d$ dimensions the count grows like $\epsilon^{-d}$.
- *whether gradient descent finds* the approximant.
- *how many samples* identify it, or *which* of the many interpolants of the data training selects.

Polynomials (Weierstrass) and lookup tables are universal approximators too. Expressivity is cheap, so it cannot be what distinguishes networks. UAT rules out "the net cannot represent the target" and nothing else.

### 2.C. Capacity bounds, and why they are vacuous here

§1.A.i.3 reduced the excess risk to the worst gap over the class. For a finite
class with losses in $[0,1]$, Hoeffding's inequality and a union bound give,
with probability at least $1-\delta$,

$$
\sup_{h\in\mathcal H}|R(h)-\hat R_n(h)|\le\sqrt{\frac{\ln(2|\mathcal H|/\delta)}{2n}} .
$$

For infinite classes of binary classifiers, $\ln|\mathcal H|$ is replaced by a
combinatorial dimension. When $n$ is sufficiently larger than the VC dimension $d$, the bound has the shape
$\sqrt{(d\ln(n/d)+\ln(1/\delta))/n}$ up to constants (quoted, not proved).
For ReLU networks with $W$ weights in $L$ layers, the general upper bound is
$O(WL\log W)$, with constructions achieving $\Omega(WL\log(W/L))$;
this does not assign that dimension to every architecture. See
[Bartlett et al. (2019)](https://jmlr.org/papers/v20/17-612.html). Finite
classes and VC dimension are in UML ch. 4 and ch. 6.

#### 2.C.i. Plug in a real network

1. Derive the finite-class bound from Hoeffding's inequality, $\Pr[|\hat R_n(h)-R(h)|>t]\le2e^{-2nt^2}$, and a union bound.
2. A network with $W=10^6$ float32 weights is a finite class. Evaluate the bound for $n=5\times10^4$ (the size of the CIFAR-10 training set) and $\delta=0.05$. Then compare $d\approx W\ln W$ with $n$.
3. Zhang et al. (below) show that this kind of architecture, trained by SGD, fits CIFAR-10 images with uniformly random labels to zero training error. Let $\mathcal H$ be the set of functions the architecture and SGD can produce. Assume such interpolation succeeds with high probability on i.i.d. samples from a distribution with independent uniform labels and distinct inputs. Show that every bound of the form "for every distribution $P$, with probability $\ge1-\delta$, all $h\in\mathcal H$ have $\mathrm{gap}(h)\le\epsilon(n,\delta)$" must have $\epsilon\ge0.9$ (0–1 loss, 10 classes), up to sampling fluctuation.

<details><summary>Hint</summary>

For 3, the bound has to hold for *every* $P$. Choose one.
</details>

**1.** For a fixed $h$, $\Pr[|\hat R_n(h)-R(h)|>t]\le2e^{-2nt^2}$. A union bound over $|\mathcal H|$ hypotheses gives $\Pr[\sup_h|\cdot|>t]\le2|\mathcal H|e^{-2nt^2}$. Setting this equal to $\delta$ and solving gives $t=\sqrt{\ln(2|\mathcal H|/\delta)/(2n)}$.

**2.** Using the parameter-encoding upper bound $\ln|\mathcal H|\le32W\ln2\approx2.22\times10^7$, so the bound is $\sqrt{(2.22\times10^7+\ln 40)/10^5}\approx\sqrt{222}\approx14.9$. Since the 0–1 loss is at most 1, this parameter-encoding bound says nothing. The rough VC scale is $W\ln W\approx1.4\times10^7\gg n=5\times10^4$. The VC route needs $n\gg d$, so a dimension of that size would make it uninformative too. This scale comes from the generic upper bound; it does not prove that the actual architecture has that VC dimension. If the actual VC dimension is at least $n$, the class shatters some $n$ points. Under uniform sampling on those points, a random target leaves an expected unseen mass $(1-1/n)^n$, and NFL gives expected error at least $\tfrac12(1-1/n)^n$ for some target. Thus uniformly small error is impossible at that sample size; this is not a claim that every distribution-free bound is impossible.

**3.** Take $P'$ to be CIFAR-10 images with labels drawn uniformly from the 10 classes, independently of the image. Assume the random-label fitting result extends to i.i.d. samples from this input distribution, with distinct inputs and high probability of interpolation. Then SGD reaches an $h\in\mathcal H$ with $\hat R_n(h)=0$. Its labels are independent of the input, so $R(h)=0.9$. So on $P'$, some $h\in\mathcal H$ has gap $0.9$ with high probability. A bound that depends only on $\mathcal H$, $n$ and $\delta$ must hold for $P'$, so $\epsilon(n,\delta)\ge0.9-o(1)$. The same distribution-independent bound is then vacuous for the real labels as well. The experiment on one fixed image collection motivates the interpolation assumption; it does not prove it for every random draw. Vacuous is not wrong, but it cannot be the explanation. 2.D.i.1 asks which explanations survive.

### 2.D. Reading

**[Understanding deep learning requires rethinking generalization](https://arxiv.org/abs/1611.03530)**, Chiyuan Zhang, Samy Bengio, Moritz Hardt, Benjamin Recht, Oriol Vinyals (ICLR 2017) · *core*

Standard image-classification networks trained by SGD fit random labels, and even random pixels, to zero training error, with or without explicit regularization. So capacity measures of the model class cannot explain why the same networks generalize on real labels. Read §1 (the claim and the contributions) and §2 (the randomization tests: true labels, partially and fully random labels, shuffled and random pixels; how training time changes). §3 (the role of explicit and implicit regularization) is short and worth skimming. The finite-sample expressivity construction (§4) and the linear-model discussion (§5) are optional.

#### 2.D.i. Questions on Zhang et al.

1. List the explanations of generalization that the random-labels experiment rules out, and those it leaves standing.
2. The paper finds that explicit regularizers (weight decay, dropout, data augmentation) help but are neither necessary nor sufficient. Which of §4's three views of regularization (prior, constraint, spectral filter) does this count against, and which survive? (This uses §4.)

**1.** *Ruled out:*
- any explanation through small capacity of the reachable class that holds for every distribution (VC dimension, Rademacher complexity of the class; as the paper notes in §2.2, fitting random labels means the empirical Rademacher complexity is essentially 1);
- explicit regularization as the main cause, since the nets generalize nearly as well without it;
- "SGD cannot find memorizing solutions", since it finds them, only a constant factor more slowly.

*Not ruled out:*
- data-dependent accounts, in which the learned function depends on structure in the true labels: an implicit bias toward low-norm, flat or simple solutions *when such solutions exist*;
- norm-, margin-, compression- and PAC-Bayes-style bounds evaluated on the learned function;
- a prior over functions whose support is everything but whose mass is on structured functions.

The slower fit on random labels is a hint that some simplicity ordering is at work. Such a "soft" bias (the class can fit anything, but training prefers some fits) predicts things a hard capacity restriction does not:
- the same model at the same $n$ generalizes differently depending on the labels;
- random labels take longer to fit and end at larger norm or complexity;
- adding parameters need not hurt, and can help (§5), whereas a class-wide capacity upper bound can worsen with class size without predicting that actual risk must worsen;
- a bound in terms of the prior mass or description length of the *found* function can be non-vacuous where class-capacity bounds are not.

**2.** It counts against the *constraint* view used as an explanation, in which a norm ball is small enough for its capacity to explain generalization. The nets generalize without the constraint, and with the constraint they still fit random labels, so the constrained class is still huge. The *prior* view survives if the prior is soft and implemented implicitly, by architecture, initialization and SGD. The *spectral-filter* view also survives as implicit regularization: early stopping and gradient descent's preference for large-eigenvalue directions, and gradient descent's convergence to the minimum-norm solution (§4.C).

<a id="wb2-section-3"></a>

## 3. Bias–variance

**The result.** For squared loss, the expected error at a point splits exactly
into noise, squared bias and variance. The same algebra shows that a single
random member of an ensemble, or a single posterior draw, does worse than the
average prediction by the spread of the members' predictions. Ridge regression makes the bias–variance
split explicit in the eigenbasis of $X^\top X$. Direction $k$ is shrunk by
$s_k^2/(s_k^2+n\lambda)$, which trades variance in weak directions for bias.
Some ridge always beats none, and the best penalty in direction $k$ is
$n\lambda_k=\sigma^2/\beta_k^2$, whatever $s_k$ is. The effective number of
parameters is $\sum_k s_k^2/(s_k^2+n\lambda)$.

### 3.A. The pointwise decomposition

Let $y=f(x)+\varepsilon$ with $\mathbb E[\varepsilon\mid x]=0$ and
$\mathrm{Var}[\varepsilon\mid x]=\sigma^2(x)$. An estimator $\hat f_S$ is trained
on a random sample $S$, independent of the test point. Fix $x$.

*Source:* ESL §7.3 (the decomposition) and §3.4.1 (ridge in the singular-value
basis, and its effective degrees of freedom).

#### 3.A.i. Decompose, then decompose a posterior

1. Show that $\mathbb E_{S,\varepsilon}\big(\hat f_S(x)-y\big)^2=\sigma^2(x)+\big(\mathbb E_S\hat f_S(x)-f(x)\big)^2+\mathrm{Var}_S\hat f_S(x)$.
2. Average $K$ predictors trained on independent samples. Which terms change, and by how much?
3. The same algebra with a different source of randomness. Let $q$ be any distribution over parameters, for instance a posterior given $S$, with predictive mean $\bar f(x)=\mathbb E_{\theta\sim q}f_\theta(x)$. Show that for squared loss
$$\mathbb E_{\theta\sim q}\big(f_\theta(x)-y\big)^2=\big(\bar f(x)-y\big)^2+\mathrm{Var}_{\theta\sim q}f_\theta(x).$$
Translate this into a statement comparing a single posterior draw with the posterior-predictive mean. (§7.E calls their errors the Gibbs and Bayes errors.)
4. Does 1 hold for 0–1 loss? Give an example where *more* variance lowers the expected 0–1 error.

**1.** Write $\hat f_S(x)-y=(\hat f_S-\mathbb E\hat f_S)+(\mathbb E\hat f_S-f)+(f-y)$ and square. The cross terms vanish in expectation. $(\hat f_S-\mathbb E\hat f_S)$ has mean zero over $S$. $(f-y)=-\varepsilon$ has mean zero and is independent of $S$. $(\mathbb E\hat f_S-f)$ is a constant. What remains is variance, plus squared bias, plus $\sigma^2(x)$.

**2.** The average of $K$ independent copies has the same mean, hence the same bias, and variance $\mathrm{Var}_S\hat f_S(x)/K$. The noise term is untouched. (Real bagging resamples a single dataset, so the copies are correlated and the reduction is smaller.)

**3.** Expand $f_\theta-y=(f_\theta-\bar f)+(\bar f-y)$ and take $\mathbb E_{\theta\sim q}$ with $x,y$ fixed. The cross term is $2(\bar f-y)\,\mathbb E_q[f_\theta-\bar f]=0$. With $q$ a posterior, the left side is the Gibbs loss at $(x,y)$ and $(\bar f-y)^2$ is the Bayes-ensemble loss. Averaging over $(x,y)$:

$$
\text{Gibbs error}=\text{Bayes-ensemble error}+\mathbb E_x\,\mathrm{Var}_{\text{post}}f_\theta(x)\ \ge\ \text{Bayes-ensemble error},
$$

with equality iff the posterior predictors agree for almost every input under the test distribution. For squared loss, ensembling never hurts a posterior draw. §7 finds that in the regular case the excess parts differ by a factor of two ($p/n$ vs $p/2n$).

**4.** No: the 0–1 loss does not decompose additively. Take $y=1$ deterministically and predict $\mathbf 1[\hat g_S(x)>0]$, where $\hat g_S(x)\sim\mathcal N(\mu,v)$ with $\mu<0$, so the average prediction is on the wrong side. The error is $\Pr[\hat g_S\le0]=\Phi(-\mu/\sqrt v)$, which *decreases* toward $\tfrac12$ as $v$ grows. When the bias has the wrong sign, variance helps. "Variance always hurts" is a fact about squared loss.

### 3.B. Ridge in the eigenbasis

Use a fixed design and noise $\varepsilon\sim(0,\sigma^2I_n)$ and $\sigma^2>0$. For the comparisons with OLS below assume $X$ has full column rank; the positive-penalty formulas also apply to rank-deficient designs. Ridge regression is

$$
\hat\beta_\lambda=\arg\min_b\ \tfrac1n\|y-Xb\|^2+\lambda\|b\|^2=(X^\top X+n\lambda I)^{-1}X^\top y .
$$

Write $X^\top X=V\,\mathrm{diag}(s_1^2,\dots,s_p^2)\,V^\top$, where the $s_k$ are
the singular values of $X$, and $\beta_k=v_k^\top\beta$. Write $a=n\lambda$ for
short. We track the parameter error $\mathbb E\|\hat\beta_\lambda-\beta\|^2$,
which equals the excess population risk when $x\sim\mathcal N(0,I)$ (§1.B).

#### 3.B.i. Bias and variance per direction

1. Show that, in coordinates $v_k^\top\hat\beta_\lambda$,
$$\mathrm{bias}_k=-\frac{a}{s_k^2+a}\,\beta_k,\qquad \mathrm{var}_k=\sigma^2\frac{s_k^2}{(s_k^2+a)^2},$$
and that different coordinates are uncorrelated. Hence $\mathbb E\|\hat\beta_\lambda-\beta\|^2=\sum_k\dfrac{a^2\beta_k^2+\sigma^2s_k^2}{(s_k^2+a)^2}$.
2. Which directions carry most of the variance at $\lambda=0$? Show that the derivative of the total error with respect to $a$ at $a=0$ is strictly negative, so some ridge always beats OLS.
3. Allow a separate penalty $a_k$ in each direction. Show that the optimum is $a_k=\sigma^2/\beta_k^2$, independent of $s_k$, and that the minimal error in that direction is $\big(s_k^2/\sigma^2+1/\beta_k^2\big)^{-1}$. If $\beta\sim\mathcal N(0,\tau^2I)$, show that the best *common* penalty is $a=\sigma^2/\tau^2$.
4. *Effective degrees of freedom.* The fitted values are $\hat y=S_\lambda y$ with $S_\lambda=X(X^\top X+aI)^{-1}X^\top$. Show that $\mathrm{Tr}\,S_\lambda=\mathrm{df}(\lambda)=\sum_k s_k^2/(s_k^2+a)$. Then show that the in-sample optimism, $\mathbb E\frac1n\|y'-\hat y\|^2-\mathbb E\frac1n\|y-\hat y\|^2$ for fresh noise $y'=X\beta+\varepsilon'$ at the same inputs, equals $2\sigma^2\,\mathrm{df}(\lambda)/n$. Compare with §1.B.ii.

<details><summary>Hint</summary>

For 1, $\mathbb E\hat\beta_\lambda=(X^\top X+aI)^{-1}X^\top X\beta$, and the covariance is $\sigma^2(X^\top X+aI)^{-1}X^\top X(X^\top X+aI)^{-1}$. All of these matrices are diagonal in $V$. For 3, differentiate $f(a)=(a^2\beta^2+\sigma^2s^2)/(s^2+a)^2$; the numerator of $f'$ factors. For 4, expand both squared norms around $X\beta$. The only surviving difference is $\mathbb E\langle\varepsilon,S_\lambda\varepsilon\rangle$.
</details>

**1.** $\mathbb E\hat\beta_\lambda=(X^\top X+aI)^{-1}X^\top X\beta$, so in the $V$ basis coordinate $k$ is $\frac{s_k^2}{s_k^2+a}\beta_k$ and the bias is $\big(\frac{s_k^2}{s_k^2+a}-1\big)\beta_k=-\frac{a}{s_k^2+a}\beta_k$. The covariance is $\sigma^2(X^\top X+aI)^{-1}X^\top X(X^\top X+aI)^{-1}=V\,\mathrm{diag}\!\big(\sigma^2s_k^2/(s_k^2+a)^2\big)V^\top$, which is diagonal in $V$, so the coordinates are uncorrelated. Summing squared bias and variance over $k$ gives the total.

**2.** At $a=0$, $\mathrm{var}_k=\sigma^2/s_k^2$, so the weakest directions (small $s_k$) dominate. That is the same $\mathrm{Tr}(X^\top X)^{-1}$ as in §1.B. Write $f_k(a)=\frac{a^2\beta_k^2+\sigma^2s_k^2}{(s_k^2+a)^2}$. Then

$$
f_k'(a)=\frac{2a\beta_k^2(s_k^2+a)-2(a^2\beta_k^2+\sigma^2s_k^2)}{(s_k^2+a)^3}=\frac{2s_k^2(a\beta_k^2-\sigma^2)}{(s_k^2+a)^3}.
$$

At $a=0$ this is $-2\sigma^2/s_k^4<0$ for every $k$, so a small amount of ridge lowers the total error, for every $\beta$. This is the observation of [Hoerl and Kennard (1970)](https://doi.org/10.1080/00401706.1970.10488634).

**3.** By the factored derivative, $f_k$ decreases for $a<\sigma^2/\beta_k^2$ and increases after, so $a_k^*=\sigma^2/\beta_k^2$, which does not involve $s_k$. Substituting,

$$
f_k(a_k^*)=\frac{\sigma^2(\sigma^2+s_k^2\beta_k^2)/\beta_k^2}{(s_k^2\beta_k^2+\sigma^2)^2/\beta_k^4}=\frac{\sigma^2\beta_k^2}{s_k^2\beta_k^2+\sigma^2}=\Big(\frac{s_k^2}{\sigma^2}+\frac1{\beta_k^2}\Big)^{-1}.
$$

If $\beta_k=0$, the optimum is the limit $a_k\to\infty$, with zero error. This has the form of a posterior variance: data precision plus prior precision. If $\beta\sim\mathcal N(0,\tau^2I)$, then $\mathbb E\beta_k^2=\tau^2$, and $\mathbb E_\beta f_k(a)=\frac{a^2\tau^2+\sigma^2s_k^2}{(s_k^2+a)^2}$ is minimized at $a=\sigma^2/\tau^2$ for every $k$ at once. So the best common penalty is $n\lambda=\sigma^2/\tau^2$. §4.A gets the same value as the MAP prior, and §5 as the optimal $\lambda^*=\sigma^2\gamma/r^2$, with $\tau^2=r^2/p$.

**4.** With the thin SVD $X=U\,\mathrm{diag}(s_k)V^\top$, $S_\lambda=U\,\mathrm{diag}\big(s_k^2/(s_k^2+a)\big)U^\top$, so $\mathrm{Tr}\,S_\lambda=\sum_k s_k^2/(s_k^2+a)\in[0,p]$. Let $\mu=X\beta$. Then

$$
\|y'-\hat y\|^2=\|\varepsilon'\|^2+\|\mu-\hat y\|^2+2\langle\varepsilon',\mu-\hat y\rangle,\qquad
\|y-\hat y\|^2=\|\varepsilon\|^2+\|\mu-\hat y\|^2+2\langle\varepsilon,\mu-\hat y\rangle .
$$

The first cross term has mean $0$ because $\varepsilon'$ is independent of $\hat y$. In the second, $\mu-\hat y=(I-S_\lambda)\mu-S_\lambda\varepsilon$, so $\mathbb E\langle\varepsilon,\mu-\hat y\rangle=-\sigma^2\,\mathrm{Tr}\,S_\lambda$. The difference is $2\sigma^2\,\mathrm{Tr}\,S_\lambda$, and dividing by $n$ gives $2\sigma^2\mathrm{df}(\lambda)/n$. At $\lambda=0$, $\mathrm{df}=p$, which has the same leading $2\sigma^2p/n$ value as §1.B.ii. Here it is exact for new labels at the training inputs; §1.B.ii evaluates fresh random inputs, so its finite-sample gap differs. Ridge replaces "count the parameters" with "count the directions the data can resolve against the penalty."

The textbook U-shaped curve, where test error falls with model size and then
rises, is §3.B with the penalty replaced by the number of features. §5 shows
the U is not the whole story even in least squares.

<a id="wb2-section-4"></a>

## 4. Regularization three ways, and how not to think about it

**The result.** In linear regression, ridge is three things at once: the MAP
estimate and posterior mean under a Gaussian prior, the Lagrangian form of a
norm constraint, and a filter that shrinks each eigendirection of the data by
$s_k^2/(s_k^2+n\lambda)$. Early-stopped gradient flow is a related filter,
$1-e^{-s_k^2t/n}$, but not the same one: the half-points match at
$t=\ln2/\lambda$, not $1/\lambda$. Dropout is ridge in a data-dependent
diagonal metric. Flatness can matter, because the mean excess risk is locally approximately
$\tfrac12\mathrm{Tr}(H\Sigma)$. But sharpness on its own ($\mathrm{Tr}\,H$) is
not a property of the learned function, so any flatness claim has to say which
metric it uses.

Setup, as in §3.B: $\hat\beta_\lambda=(X^\top X+n\lambda I)^{-1}X^\top y$,
noise variance $\sigma^2$, and the thin SVD $X=U\,\mathrm{diag}(s_k)\,V^\top$
with left singular vectors $u_k$ and right singular vectors $v_k$.

### 4.A. As a prior

*Source:* ESL §3.4.1 notes that ridge is the posterior mode and mean under a
Gaussian prior.

#### 4.A.i. MAP, posterior, and a warning

1. Take the prior $\beta\sim\mathcal N(0,\tau^2I)$ and the likelihood $y\mid\beta\sim\mathcal N(X\beta,\sigma^2I)$. Show that the MAP estimate is ridge with $\lambda=\sigma^2/(n\tau^2)$. Show that the posterior is $\mathcal N\big(\hat\beta_\lambda,\ \sigma^2(X^\top X+n\lambda I)^{-1}\big)$, so the posterior mean equals the MAP here.
2. Compare the posterior covariance with the sampling covariance of $\hat\beta_\lambda$ from §3.B. Which is larger, and why do they differ?
3. MAP depends on the parametrization. Suppose a posterior on $\beta>0$ has density $\pi(\beta)\propto\beta e^{-\beta}$. Find its mode. Now write $\beta=e^u$, find the mode of the density of $u$, and map it back. Which of the mode, mean and median is invariant under monotone reparametrization? What does this mean for "weight decay is a prior"?

**1.** The negative log posterior is $\frac1{2\sigma^2}\|y-X\beta\|^2+\frac1{2\tau^2}\|\beta\|^2+\text{const}$. Multiplying by $2\sigma^2/n$ gives $\frac1n\|y-X\beta\|^2+\frac{\sigma^2}{n\tau^2}\|\beta\|^2$, the ridge objective with $\lambda=\sigma^2/(n\tau^2)$. The log posterior is quadratic in $\beta$, so the posterior is Gaussian. Its precision is $X^\top X/\sigma^2+I/\tau^2=(X^\top X+n\lambda I)/\sigma^2$, and its mean (which is also the mode) is $\hat\beta_\lambda$.

**2.** Write $G=X^\top X$ and $a=n\lambda$. The posterior covariance is $\sigma^2(G+aI)^{-1}$ and the sampling covariance is $\sigma^2G(G+aI)^{-2}$. Their difference is $\sigma^2a(G+aI)^{-2}\succeq0$, so the posterior is wider. The sampling covariance ignores the bias. The posterior covariance is the bias–variance mean squared error averaged over the prior: per direction, at $a=\sigma^2/\tau^2$,

$$
\frac{a^2\tau^2+\sigma^2s^2}{(s^2+a)^2}=\frac{\sigma^2(a+s^2)}{(s^2+a)^2}=\frac{\sigma^2}{s^2+a}.
$$

**3.** $\frac{d}{d\beta}\log(\beta e^{-\beta})=1/\beta-1=0$, so the mode is $\beta=1$. The density of $u=\log\beta$ is $\pi(e^u)e^u\propto e^{2u-e^u}$, and $\frac{d}{du}(2u-e^u)=0$ gives $e^u=2$, i.e. $\beta=2$. Same posterior, different "MAP". The mean is not invariant under nonlinear maps either. The median is invariant under monotone maps.

So the identification "penalty = negative log prior" holds *in a chosen parametrization*. Weight decay is a Gaussian prior on the weights, which in a network induces some prior on functions that depends on the architecture. The MAP estimate under it is not an invariant summary of the posterior.

### 4.B. As a constraint

#### 4.B.i. The Lagrange multiplier and its monotone map

1. For $c>0$, show that a minimizer of $\hat R_n(b)=\frac1n\|y-Xb\|^2$ subject to $\|b\|^2\le c$ is $\hat\beta_\lambda$, where $\lambda$ is the KKT multiplier. Interpret $\hat\beta_0=X^+y$ when the design is rank deficient.
2. Show that $c(\lambda)=\|\hat\beta_\lambda\|^2$ is strictly decreasing when $X^\top y\neq0$. Hence $\lambda(c)$ is well defined for $c<\|X^+y\|^2$, and $\lambda=0$ for $c\ge\|X^+y\|^2$.
3. In this view, what explains why the constrained estimator generalizes? Which objection from §2 applies to that explanation?

<details><summary>Hint</summary>

For 1, write the Lagrangian $\hat R_n(b)+\lambda(\|b\|^2-c)$ with $\lambda\ge0$, set its gradient to zero, and use complementary slackness. For 2, write $\|\hat\beta_\lambda\|^2$ in the SVD of $X$ and look at each term as a function of $\lambda$. For 3, recall §1.A.i.3 and §2.C; the Rademacher complexity of a norm ball of linear predictors is in UML §26.2.
</details>

**1.** The problem is convex, and Slater's condition holds for $c>0$. The Lagrangian is $\hat R_n(b)+\lambda(\|b\|^2-c)$ with $\lambda\ge0$. Stationarity gives $-\frac2nX^\top(y-Xb)+2\lambda b=0$, i.e. $b=(X^\top X+n\lambda I)^{-1}X^\top y=\hat\beta_\lambda$ for $\lambda>0$. For $\lambda=0$, use the minimum-norm least-squares solution $X^+y$; other feasible least-squares solutions may also minimize when $X$ is rank deficient. Complementary slackness says either $\lambda=0$ (the constraint need not be strictly slack) or $\|\hat\beta_\lambda\|^2=c$.

**2.** In the SVD, $\|\hat\beta_\lambda\|^2=\sum_k\frac{s_k^2(u_k^\top y)^2}{(s_k^2+n\lambda)^2}$. Each term is non-increasing in $\lambda$, and at least one is strictly decreasing when $X^\top y\ne0$. As $\lambda\downarrow0$ the norm rises to $\|X^+y\|^2$, the minimum-norm least-squares solution, and as $\lambda\to\infty$ it falls to $0$. So the map from $c$ to $\lambda$ is a decreasing bijection on $(0,\|X^+y\|^2)$. Tighter constraints need bigger multipliers.

**3.** The explanation is capacity. The norm ball $\{x\mapsto x^\top b:\|b\|\le B\}$ has Rademacher complexity of order $B\sqrt{\mathbb E\|x\|^2/n}$ (quoted), which controls prediction complexity. To turn this into a squared-loss gap bound, additionally control the input norms and labels (or suitable tails), so the loss is Lipschitz on the allowed prediction range. Small enough complexity then gives an excess-risk bound via §1.A.i.3. This inherits two problems. The norm depends on the parametrization (4.A.i.3, 4.E), and Zhang et al.'s objection applies: nets with weight decay still fit random labels, so the constraint that is actually active is not small.

### 4.C. As a spectral filter, and early stopping

#### 4.C.i. Two filters

1. Show that $\hat\beta_\lambda=\sum_{k:\,s_k>0}\phi_k\,\dfrac{u_k^\top y}{s_k}\,v_k$ with $\phi_k^{\mathrm{ridge}}=\dfrac{s_k^2}{s_k^2+n\lambda}$. The factor $(u_k^\top y)/s_k$ is the unregularized coefficient.
2. Run gradient flow $\dot\beta=-\nabla L(\beta)$ on $L(\beta)=\frac1{2n}\|X\beta-y\|^2$, starting from $\beta(0)=0$. Show that $\beta(t)$ has the same form with $\phi_k^{\mathrm{GF}}(t)=1-e^{-s_k^2t/n}$. For gradient descent with step $\eta$ and $T$ steps, show $\phi_k=1-(1-\eta s_k^2/n)^T$.
3. Show that the two filters cross $\tfrac12$ at the same $s^2$ if and only if $t=\ln2/\lambda$. With that matching, write both in terms of $x=s^2/(n\lambda)$, and compare them for $x\ll1$ and for $x\gg1$. Which filter passes more of the weak directions? Which shrinks the strong directions more?
4. Let $p>n$. Show that gradient flow from $\beta(0)=0$ converges to the minimum-norm interpolant $X^+y$. What is the limit from a nonzero initialization $\beta_0$?

<details><summary>Hint</summary>

Work in the coordinates $b_k=v_k^\top\beta$, where $X^\top X$ is diagonal and $X^\top y=\sum_ks_k(u_k^\top y)v_k$; gradient flow then decouples into one scalar linear ODE per $k$. For 3, set each filter equal to $\tfrac12$ and solve for $s^2$. For 4, in which subspace does $\dot\beta$ always lie?
</details>

*Further reading:* [Ali, Kolter and Tibshirani (2019)](https://arxiv.org/abs/1810.10082)
compare the risks of gradient flow and ridge along the whole path, under the
calibration $t=1/\lambda$.

**1.** $X^\top y=\sum_k s_k(u_k^\top y)v_k$ and $(X^\top X+n\lambda I)^{-1}v_k=v_k/(s_k^2+n\lambda)$, so $\hat\beta_\lambda=\sum_k\frac{s_k(u_k^\top y)}{s_k^2+n\lambda}v_k=\sum_k\frac{s_k^2}{s_k^2+n\lambda}\cdot\frac{u_k^\top y}{s_k}v_k$.

**2.** $\nabla L=\frac1n(X^\top X\beta-X^\top y)$. In coordinates $b_k=v_k^\top\beta$,

$$
\dot b_k=-\frac{s_k^2}{n}b_k+\frac{s_k}{n}u_k^\top y,
$$

with fixed point $(u_k^\top y)/s_k$. From $b_k(0)=0$ the solution is $b_k(t)=(1-e^{-s_k^2t/n})(u_k^\top y)/s_k$. For gradient descent, $b_k^{(T)}-b_k^\infty=(1-\eta s_k^2/n)^T(b_k^{(0)}-b_k^\infty)$, which gives the stated filter. Directions with $s_k=0$ never move. Discrete gradient descent converges on the positive eigendirections when $0<\eta<2n/s_{\max}^2$; the filter is non-oscillatory for $\eta\le n/s_{\max}^2$.

**3.** Ridge equals $\tfrac12$ at $s^2=n\lambda$, and flow equals $\tfrac12$ at $s^2t/n=\ln2$, i.e. at $s^2=n\ln2/t$. These coincide iff $t=\ln2/\lambda$. With this matching, ridge is $x/(1+x)$ and flow is $1-e^{-x\ln2}=1-2^{-x}$.
- *Weak directions,* $x\ll1$: ridge $\approx x$ and flow $\approx x\ln2$, so ridge passes about $1/\ln2\approx1.44$ times more of each weak direction.
- *Strong directions,* $x\gg1$: ridge leaves a shrinkage of $1-\phi\approx1/x$, only polynomially small, while flow leaves $2^{-x}$, exponentially small. So ridge shrinks the strong directions more.

Gradient flow is the sharper, more step-like filter. The largest disagreements are flow above ridge by about $0.138$ (near $x\approx4.3$) and ridge above flow by about $0.044$ (near $x\approx0.36$). With the naive matching $t=1/\lambda$, flow becomes $1-e^{-x}$ and its half-point moves to $x=\ln2$. "Early stopping is ridge" is right in outline (keep the strong directions, drop the weak ones) and wrong in detail. WB5 §7 derives how the polynomial tail of the ridge filter affects learning-curve exponents.

**4.** $\dot\beta\in\mathrm{row}(X)$, so $\beta(t)\in\mathrm{row}(X)$ for all $t$. For $p>n$ with $X$ of full row rank, all $s_k>0$ for $k\le n$, so by 2 the flow converges to $\sum_{k\le n}\frac{u_k^\top y}{s_k}v_k=X^+y$. That is the unique interpolant in the row space, and so the minimum-norm interpolant. From $\beta_0$, the row-space component converges to the same place and the null-space component never moves: the limit is $X^+y+(I-X^+X)\beta_0$. The implicit bias is "minimum distance from the initialization".

▶ **Interactive figure:** [`spectral_filters.html`](assets/interactive/spectral_filters.html). Use panel A only (panel B belongs to WB5). Press “match half-points” and read the difference strip: where do the filters disagree, and which passes more of the weak tail? Then set t = 1/λ by hand and see the ln 2 offset.

### 4.D. Dropout as adaptive $L_2$ (optional)

Drop inputs independently: $\tilde x_{ij}=x_{ij}\xi_{ij}/q$ with
$\xi_{ij}\sim\mathrm{Bernoulli}(q)$, where $q$ is the keep probability. The
$1/q$ keeps $\mathbb E\tilde x_{ij}=x_{ij}$. Part 3 is the main observation of
[Wager, Wang and Liang (2013)](https://arxiv.org/abs/1307.1493).

#### 4.D.i. The penalty hiding in the noise

1. Show that $\mathbb E_\xi\frac1n\sum_i(y_i-\tilde x_i^\top\beta)^2=\frac1n\|y-X\beta\|^2+\frac{1-q}{q}\sum_j\beta_j^2\,\frac{\|x_{\cdot j}\|^2}{n}$, where $x_{\cdot j}$ is the $j$-th column of $X$.
2. Write the penalty as $\beta^\top\Gamma\beta$. Show that it is invariant under rescaling a feature ($x_{\cdot j}\to cx_{\cdot j}$, $\beta_j\to\beta_j/c$), while the ridge penalty is not. Which standard procedure is it equivalent to?
3. Take a generalized linear model with per-example loss $A(x^\top\beta)-y\,x^\top\beta$ (for logistic regression, $A(\eta)=\log(1+e^\eta)$). Expand to second order in the dropout noise and show that the expected penalty is approximately $\frac{1-q}{2q}\cdot\frac1n\sum_iA''(x_i^\top\beta)\sum_jx_{ij}^2\beta_j^2$. In what sense is this "adaptive"?

<details><summary>Hint</summary>

Write $\tilde x_i^\top\beta=x_i^\top\beta+\sum_jx_{ij}\beta_j(\xi_{ij}/q-1)$. The noise has mean zero and $\mathrm{Var}(\xi/q)=(1-q)/q$.
</details>

**1.** Let $\eta_i=x_i^\top\beta$ and $\delta_i=\tilde x_i^\top\beta-\eta_i=\sum_jx_{ij}\beta_j(\xi_{ij}/q-1)$. Then $\mathbb E\delta_i=0$ and, by independence, $\mathrm{Var}\,\delta_i=\frac{1-q}{q}\sum_jx_{ij}^2\beta_j^2$. So $\mathbb E(y_i-\eta_i-\delta_i)^2=(y_i-\eta_i)^2+\mathrm{Var}\,\delta_i$. Averaging over $i$ gives the penalty $\frac{1-q}{q}\sum_j\beta_j^2\|x_{\cdot j}\|^2/n$.

**2.** $\Gamma=\frac{1-q}{q}\,\mathrm{diag}(X^\top X/n)$. Under $x_{\cdot j}\to cx_{\cdot j}$, $\beta_j\to\beta_j/c$, the product $\beta_j^2\|x_{\cdot j}\|^2$ is unchanged, while ridge's $\beta_j^2$ changes by $1/c^2$. Dropout on inputs is ridge after standardizing each feature by its root mean square: ridge in the metric $\mathrm{diag}(\hat\Sigma)$.

**3.** $\mathbb E\tilde x_i^\top\beta=\eta_i$, so the linear term $-y\,\tilde x_i^\top\beta$ is unchanged in expectation. A second-order expansion gives $\mathbb E A(\eta_i+\delta_i)\approx A(\eta_i)+\tfrac12A''(\eta_i)\mathrm{Var}\,\delta_i$, so the penalty is $\frac1n\sum_i\tfrac12A''(\eta_i)\frac{1-q}{q}\sum_jx_{ij}^2\beta_j^2$. For logistic regression $A''(\eta)=\hat p(1-\hat p)$, the Fisher information of the example. The penalty is ridge in the metric of the diagonal Fisher information. Confidently classified examples contribute little, and features that matter on uncertain examples are penalized most. "Adaptive" because the metric depends on the current $\beta$. For squared loss written as $\tfrac12(y-\eta)^2$, $A''=1$, which recovers 1 up to the factor $\tfrac12$ of that convention.

The general lesson: many "hacky" regularizers are quadratic penalties in disguise, in some metric. The question to ask is "which metric?"

### 4.E. Why flatness could matter, and how not to think about it

#### 4.E.i. Local mean excess risk is approximately $\tfrac12\mathrm{Tr}(H\Sigma)$, and sharpness needs a metric

1. Let $\theta^*$ minimize $R$, with $H=\nabla^2R(\theta^*)$, and let the trained parameter be $\hat\theta=\theta^*+\delta$ with $\mathbb E\,\delta\delta^\top=\Sigma$. Show that $\mathbb E R(\hat\theta)-R(\theta^*)\approx\tfrac12\mathrm{Tr}(H\Sigma)$. Check that this is exact for OLS with §1.B's $\Sigma$.
2. Suppose $\hat\theta$ is ERM in a regular model with $\Sigma\approx\frac1nH^{-1}CH^{-1}$ (the sandwich, derived in §7), where $C$ is the covariance of the per-sample gradients at $\theta^*$. Show that $\tfrac12\mathrm{Tr}(H\Sigma)=\frac1{2n}\mathrm{Tr}(H^{-1}C)$, and that if $C=cH$ this equals $cp/(2n)$, whatever the curvature. What has to be true of $\Sigma$ for "flatter is better" to hold?
3. *Dinh et al.'s rescaling.* Take the two-layer scalar linear network $f(x)=w_2w_1x$ with $R(w)=\mathbb E(y-w_2w_1x)^2$ and $a=\mathbb E x^2>0$. Assume $c^*=\mathbb E[xy]/a\ne0$; the minima are the hyperbola $w_1w_2=c^*$. Compute the Hessian at a minimum, show that $\mathrm{Tr}\,H=2a(w_1^2+w_2^2)$ and that one eigenvalue is zero. Under $(w_1,w_2)\to(\alpha w_1,w_2/\alpha)$ the function does not change. Show that $\mathrm{Tr}\,H$ takes every value in $[4a|c^*|,\infty)$. Why does the same trick work in deep ReLU networks?
4. Let $\theta=g(\varphi)$ be a smooth reparametrization with Jacobian $J$ at the minimum, and let $\delta$ transform as a tangent vector. Show that $\mathrm{Tr}(H\Sigma)$ is invariant while $\mathrm{Tr}\,H$ is not. What does a meaningful flatness claim have to specify?

<details><summary>Hint</summary>

For 1, the gradient vanishes at $\theta^*$, and $\mathbb E\,\delta^\top H\delta=\mathrm{Tr}(H\,\mathbb E\,\delta\delta^\top)$. For 3, write $R=a(w_1w_2-c^*)^2+\text{const}$. For 4, at a critical point the Hessian transforms as $J^\top HJ$, because the term involving the second derivative of $g$ multiplies $\nabla R=0$.
</details>

**1.** $R(\theta^*+\delta)=R(\theta^*)+\nabla R(\theta^*)^\top\delta+\tfrac12\delta^\top H\delta+O(\|\delta\|^3)$. The gradient term vanishes, and $\mathbb E\,\delta^\top H\delta=\mathrm{Tr}(H\Sigma)$. Here $\Sigma$ is the second moment about $\theta^*$, so it includes any bias. For OLS, $R(b)=\sigma^2+\|b-\beta\|^2$ is exactly quadratic with $H=2I$, and $\Sigma=\sigma^2(X^\top X)^{-1}$, so $\tfrac12\mathrm{Tr}(H\Sigma)=\sigma^2\mathrm{Tr}(X^\top X)^{-1}$ exactly, as in §1.B.

**2.** $\tfrac12\mathrm{Tr}\big(H\cdot\tfrac1nH^{-1}CH^{-1}\big)=\tfrac1{2n}\mathrm{Tr}(CH^{-1})$. If $C=cH$, this is $cp/(2n)$. The curvature cancels: a sharp direction is pinned down more tightly by the data ($\Sigma\propto H^{-1}$), and this exactly offsets the larger cost of a given error there. "Flatter is better" needs $\Sigma$ that does not shrink in proportion as $H$ grows. One case is a perturbation scale set by something other than the data's curvature, such as SGD noise at a fixed learning rate, for which §8 finds $\Sigma\propto H^{\gamma-1}$. So "flat minima generalize" is a claim about $H$ *and* $\Sigma$ jointly.

**3.** Write $u=w_1w_2$, so $R=a(u-c^*)^2+\text{const}$. Then $\partial_{w_1}R=2a(u-c^*)w_2$, and at a minimum ($u=c^*$),

$$
H=2a\begin{pmatrix}w_2^2&w_1w_2\\ w_1w_2&w_1^2\end{pmatrix}=2a\,vv^\top,\qquad v=(w_2,w_1)^\top .
$$

The terms proportional to $u-c^*$ vanish there. So $H$ has rank one, with eigenvalue $2a(w_1^2+w_2^2)=\mathrm{Tr}\,H$ along $v$ and eigenvalue $0$ along $(w_1,-w_2)$, the direction tangent to the hyperbola. Under the rescaling, $\mathrm{Tr}\,H=2a(\alpha^2w_1^2+w_2^2/\alpha^2)\ge4a|w_1w_2|=4a|c^*|$ by AM–GM, with equality at $\alpha^2=|w_2/w_1|$, and it is unbounded as $\alpha\to0$ or $\infty$. Every value in $[4a|c^*|,\infty)$ is attained by the same function. Weight decay would pick the balanced representative. For ReLU networks, $\mathrm{ReLU}(\alpha z)=\alpha\,\mathrm{ReLU}(z)$ for $\alpha>0$, so multiplying one layer by $\alpha$ and the next by $1/\alpha$ leaves the network function unchanged. Dinh et al. use this to make any minimum arbitrarily sharp.

**4.** At a critical point, $H_\varphi=J^\top HJ$. To first order $\delta_\theta=J\delta_\varphi$, so $\Sigma_\varphi=J^{-1}\Sigma J^{-\top}$. Then $\mathrm{Tr}(H_\varphi\Sigma_\varphi)=\mathrm{Tr}(J^\top HJJ^{-1}\Sigma J^{-\top})=\mathrm{Tr}(H\Sigma)$, which is invariant, whereas $\mathrm{Tr}\,H_\varphi=\mathrm{Tr}(J^\top HJ)$ depends on $J$. A meaningful flatness claim has to say *in what metric* the perturbations are measured: either the distribution $\Sigma$ of perturbations that actually occurs, or an invariant geometry such as the Fisher (output-space) metric. Raw $\mathrm{Tr}\,H$ in weight space is a property of the parametrization, not of the function.

### 4.F. Reading

**[Sharp minima can generalize for deep nets](https://arxiv.org/abs/1703.04933)**, Laurent Dinh, Razvan Pascanu, Samy Bengio, Yoshua Bengio (ICML 2017) · *core*

Because of the rescaling symmetries of ReLU networks, a minimum can be made arbitrarily sharp under several standard sharpness definitions without changing the function computed. So flatness in weight space cannot, on its own, explain generalization. Read §2–§4: the definitions of flatness (volume ε-flatness, Hessian-based measures, ε-sharpness), the non-negative homogeneity of rectifiers, and the construction you just did in 4.E.i.3, generalized to deep rectifier networks. §5 (reparametrizing the model and the input) is short and makes the same point more generally.

#### 4.F.i. Question

Dinh et al. is often cited as "flatness is meaningless". Using 4.E.i.4, say
precisely what the paper shows and what it leaves open. Name one
flatness-like quantity it does not touch.

The paper shows that weight-space sharpness measures ($\mathrm{Tr}\,H$, the top eigenvalue, $\epsilon$-sharpness in a Euclidean ball, flat volume) are not functions of the network function, because a symmetry of the parametrization changes them arbitrarily. So none of them can be a sufficient statistic for generalization, and an empirical correlation between them and test error depends on how parametrizations happen to be distributed in practice. It leaves open quantities that are invariant under the symmetry or carry their own metric:
- $\mathrm{Tr}(H\Sigma)$ with the $\Sigma$ that training actually produces;
- sharpness measured in the Fisher or output-space metric;
- normalized or scale-invariant sharpness;
- the volume of a function-space neighbourhood (the posterior mass of a function), which is what PAC-Bayes and Bayesian accounts use.

It also leaves open whether SGD, from standard initializations, tends to land in parametrizations where raw sharpness *happens* to track one of these. That is an empirical question about the training procedure, not about the geometry.

**Where this leaves "the hacky bullshit".** Weight decay, early stopping and
dropout are, in linear models, three ways of choosing a filter on the data's
eigendirections, each in its own metric. The "prior" reading and the "capacity
constraint" reading of the same penalty make different predictions about
random labels (§2.D). Flatness enters through $\Sigma$ and $H$ together.

<a id="wb2-section-5"></a>

## 5. Overfitting and double descent

**The result.** Take least squares with isotropic Gaussian design, in the limit
$p/n\to\gamma$. The ridgeless (minimum-norm) risk is $\sigma^2\gamma/(1-\gamma)$
below the interpolation threshold and $r^2(1-1/\gamma)+\sigma^2/(\gamma-1)$
above it. The peak at $\gamma=1$ is all variance, coming from near-zero
eigenvalues of the design. Ridge at its optimal $\lambda^*=\sigma^2\gamma/r^2$
is Bayes-optimal under the isotropic Gaussian prior below, and its risk is monotone in $\gamma$. Past the
peak, when the signal-to-noise ratio exceeds one, the interpolator dips *below*
the null risk $r^2$, with its minimum at $\gamma=r/(r-\sigma)$. So
interpolating noisy data is not automatically harmful. The name "double
descent" is from [Belkin et al. (2019)](https://arxiv.org/abs/1812.11118); the
formulas are those of Hastie et al. (reading in 5.E).

**Conventions (as in Hastie et al.).** Rows $x_i\sim\mathcal N(0,I_p)$, noise
$\varepsilon\sim\mathcal N(0,\sigma^2)$, and $\|\beta\|^2=r^2$, with $r,\sigma>0$. The risk is
$\mathcal R=\mathbb E\big[(x_0^\top\hat\beta-x_0^\top\beta)^2\big]=\mathbb E\|\hat\beta-\beta\|^2$,
which *excludes* the irreducible $\sigma^2$: it is §1's excess risk. Ridge is
$(X^\top X+n\lambda I)^{-1}X^\top y$. Limits are $p,n\to\infty$ with
$p/n\to\gamma$. The null estimator $\hat\beta=0$ has risk $r^2$.

### 5.A. Below the threshold

#### 5.A.i. The left branch

1. From §1.B.ii ($\mathbb E\,\mathcal E_n=\sigma^2p/(n-p-1)$), derive the limit $\sigma^2\gamma/(1-\gamma)$. Explain the divergence in terms of the spectrum of $X^\top X/n$.
2. What is the bias of OLS for $\gamma<1$?

**1.** $\sigma^2\frac{p}{n-p-1}=\sigma^2\frac{p/n}{1-p/n-1/n}\to\sigma^2\frac{\gamma}{1-\gamma}$. The empirical spectrum of $X^\top X/n$ converges to the Marchenko–Pastur law on $[(1-\sqrt\gamma)^2,(1+\sqrt\gamma)^2]$ (quoted). The risk is $\frac{\sigma^2}{n}\sum_i1/\lambda_i(X^\top X/n)$, and as $\gamma\to1$ the lower edge tends to $0$, so the smallest eigenvalues blow up the sum. Consistently, $\frac1p\mathrm{Tr}(X^\top X/n)^{-1}\to\frac1{1-\gamma}$.

**2.** Zero: OLS is unbiased whenever $X$ has full column rank (§1.B.i). Below the threshold the whole risk is variance.

### 5.B. Above the threshold: minimum-norm interpolation

#### 5.B.i. The right branch, bias and variance

For $p>n$, $X$ has full row rank almost surely, and infinitely many $b$ interpolate the data.

1. Show that the minimum-norm least-squares solution is $\hat\beta=X^+y=X^\top(XX^\top)^{-1}y$ and that it interpolates. It is also the $\lambda\to0^+$ limit of ridge, and the limit of gradient descent from $0$ (§4.C.i.4).
2. Let $P=X^+X$ be the orthogonal projection onto the row space of $X$. Show that $\hat\beta-\beta=-(I-P)\beta+X^+\varepsilon$, with the two terms orthogonal. So $\mathcal R=\underbrace{\mathbb E\|(I-P)\beta\|^2}_{\text{conditional bias}^2}+\underbrace{\sigma^2\,\mathbb E\,\mathrm{Tr}(XX^\top)^{-1}}_{\text{variance}}$.
3. *Bias.* Show that $\mathbb E P=(n/p)I$, so that $\text{bias}^2=r^2(1-n/p)\to r^2(1-1/\gamma)$.
4. *Variance.* $XX^\top$ is an $n\times n$ Wishart matrix with $p$ degrees of freedom. Reuse §1.B.ii with the roles of $n$ and $p$ swapped to get $\sigma^2n/(p-n-1)\to\sigma^2/(\gamma-1)$.
5. Assemble the full curve for $\gamma\in(0,\infty)$. Describe it near $\gamma=1$ and as $\gamma\to\infty$.

<details><summary>Hint</summary>

For 3, use rotation invariance: $X\to XO$ for orthogonal $O\in\mathbb R^{p\times p}$, together with $\mathrm{Tr}\,P=n$. For 4, $XX^\top=\sum_{j=1}^px_{\cdot j}x_{\cdot j}^\top$, where the columns $x_{\cdot j}\in\mathbb R^n$ are i.i.d. $\mathcal N(0,I_n)$.
</details>

**1.** $XX^\top$ is invertible, so $X\hat\beta=XX^\top(XX^\top)^{-1}y=y$, and $\hat\beta$ interpolates. Any other interpolant is $\hat\beta+w$ with $Xw=0$. Since $\hat\beta\in\mathrm{row}(X)\perp\ker X$, we have $\|\hat\beta+w\|^2=\|\hat\beta\|^2+\|w\|^2$, so $\hat\beta$ is the minimum-norm interpolant.

**2.** $\hat\beta=X^+(X\beta+\varepsilon)=P\beta+X^+\varepsilon$, so $\hat\beta-\beta=-(I-P)\beta+X^+\varepsilon$. The cross term is identically zero: $(I-P)\beta$ lies in the null space and $X^+\varepsilon$ in the row space. Here "bias squared" means the squared bias conditional on $X$, then averaged over designs, rather than bias after averaging over $X$. Then $\mathbb E\|X^+\varepsilon\|^2=\sigma^2\mathrm{Tr}(X^{+\top}X^+)$, and $X^{+\top}X^+=(XX^\top)^{-1}XX^\top(XX^\top)^{-1}=(XX^\top)^{-1}$.

**3.** For orthogonal $O$, $XO$ has the law of $X$, and its row-space projection is $O^\top PO$. So $\mathbb EP=O^\top(\mathbb EP)O$ for all $O$, which forces $\mathbb EP=cI$. Taking traces, $\mathrm{Tr}\,P=n$ gives $c=n/p$. For any fixed $\beta$, $\mathbb E\|(I-P)\beta\|^2=\beta^\top(I-\mathbb EP)\beta=r^2(1-n/p)$. The interpolator sees only the $n$-dimensional projection of $\beta$ onto a random subspace, and it misses the rest.

**4.** For $p>n+1$, by §1.B.ii.1 with $(n,p)\to(p,n)$: $\mathbb E(XX^\top)^{-1}=I_n/(p-n-1)$, so the variance is $\sigma^2n/(p-n-1)\to\sigma^2/(\gamma-1)$. (Hastie et al. state this branch as part of their Theorem 1. In the isotropic Gaussian case it is a two-line consequence of the same Wishart fact.)

**5.**

$$
\mathcal R(\gamma)=\begin{cases}\sigma^2\dfrac{\gamma}{1-\gamma},&\gamma<1,\\[2mm] r^2\Big(1-\dfrac1\gamma\Big)+\dfrac{\sigma^2}{\gamma-1},&\gamma>1.\end{cases}
$$

The risk starts at $0$, rises, and diverges at $\gamma=1$ from both sides; the divergence is entirely variance. Past the threshold, the bias grows from $0$ toward $r^2$ while the variance falls from $\infty$ toward $0$. As $\gamma\to\infty$ the risk tends to $r^2$, the null risk: the projection captures a vanishing fraction of $\beta$, and the noise is spread over so many directions that it does no harm. The "second descent" is the variance falling faster than the bias rises.

### 5.C. The second descent and the cure

#### 5.C.i. When does interpolation beat doing nothing, and what removes the peak?

*Source for 3:* [Nakkiran, Venkat, Kakade and Ma (2020)](https://arxiv.org/abs/2003.01897)
prove that optimally tuned ridge is monotone in sample size and model size in
isotropic linear regression.

1. For $\gamma>1$, minimize $r^2(1-1/\gamma)+\sigma^2/(\gamma-1)$. Show that if $r>\sigma$ (signal-to-noise ratio $r^2/\sigma^2>1$), the minimum is at $\gamma=r/(r-\sigma)$, with value $2r\sigma-\sigma^2<r^2$. Show that if $r\le\sigma$, the risk decreases monotonically toward $r^2$ from above.
2. Show that $\lambda^*=\sigma^2\gamma/r^2$ is §4.A's $\lambda=\sigma^2/(n\tau^2)$ for the prior $\beta\sim\mathcal N(0,(r^2/p)I)$, which has $\mathbb E\|\beta\|^2=r^2$. Hence ridge at $\lambda^*$ is the posterior mean, the Bayes-optimal estimator under squared loss for that prior.
3. Use 2 to argue that (a) the optimal-ridge risk is at most the min-norm risk at every $\gamma$, and (b) at fixed $p$ the optimal-ridge risk is non-increasing in $n$. Deduce that the limiting optimal-ridge curve is non-decreasing in $\gamma$, so it has no peak.

<details><summary>Hint</summary>

For 3(a), both estimators are rotation-equivariant ($X\to XO$, $\beta\to O^\top\beta$), so each one's risk depends on $\beta$ only through $\|\beta\|$. For 3(b), what can a Bayes estimator with $n+1$ samples always do that guarantees it is no worse than with $n$?
</details>

**1.** $\frac{d}{d\gamma}\big[r^2(1-1/\gamma)+\sigma^2/(\gamma-1)\big]=\frac{r^2}{\gamma^2}-\frac{\sigma^2}{(\gamma-1)^2}$. This vanishes when $\frac{\gamma-1}{\gamma}=\frac\sigma r$, i.e. $\gamma^\star=\frac{1}{1-\sigma/r}=\frac{r}{r-\sigma}$, which requires $r>\sigma$. There $1-1/\gamma^\star=\sigma/r$ and $\gamma^\star-1=\sigma/(r-\sigma)$, so

$$
\mathcal R(\gamma^\star)=r\sigma+\sigma(r-\sigma)=2r\sigma-\sigma^2=r^2-(r-\sigma)^2<r^2 .
$$

The derivative is negative before $\gamma^\star$ and positive after, and $\mathcal R\to r^2$ from below as $\gamma\to\infty$. So for SNR above one, the interpolator beats the null estimator over a whole range of $\gamma$. If $r\le\sigma$, then $\frac{\gamma-1}{\gamma}<1\le\frac\sigma r$, the derivative is negative everywhere on $(1,\infty)$, and the risk falls toward $r^2$ without crossing it: the interpolator never beats predicting zero.

In this model the global minimum over all $\gamma$ is still at $\gamma\to0$, because $r^2$ is held fixed as $p$ grows, so each added feature carries less signal. Whether adding features can win globally depends on whether they carry new signal. Hastie et al. (§5) study a misspecified variant in which they do, and there, for SNR above one, the global minimum can lie at $\gamma>1$.

**2.** With $\tau^2=r^2/p$: $\lambda=\frac{\sigma^2}{n\tau^2}=\frac{\sigma^2p}{nr^2}=\frac{\sigma^2\gamma}{r^2}=\lambda^*$. By §4.A.i.1, ridge at $\lambda^*$ is the posterior mean, which minimizes $\mathbb E\|b-\beta\|^2$ over all estimators under that prior. (This is also §3.B.i.3's best common penalty $n\lambda=\sigma^2/\tau^2$.)

**3.** (a) Ridge and min-norm are both rotation-equivariant, and the design is rotation-invariant, so each one's risk at fixed $\beta$ depends only on $\|\beta\|$. Averaged over the prior, ridge($\lambda^*$) is no worse than any estimator. For either of these linear estimators, the noise- and design-averaged risk has the form $A\|\beta\|^2+B$: isotropy makes the squared-bias matrix a scalar multiple of $I$, and the noise variance is independent of $\beta$. Thus averaging over a prior with $\mathbb E\|\beta\|^2=r^2$ gives exactly its risk at any fixed vector of norm $r$, already at finite $p$ (when the risks are finite). The Bayes comparison therefore gives the fixed-radius comparison too. This does not make ridge Bayes-optimal under the uniform-on-a-sphere prior among all estimators.

(b) A Bayes estimator given $n+1$ samples can ignore the last one, so its Bayes risk is at most that of the Bayes estimator with $n$ samples. More data never hurts the posterior mean. The limiting curve depends on $\gamma$ alone, and at fixed $p$ increasing $n$ decreases $\gamma$, so the optimal-ridge risk is non-decreasing in $\gamma$. There is no peak.

The double-descent peak is therefore a property of the minimum-norm estimator, a variance divergence at $p\approx n$ that properly tuned regularization removes, and not a property of the problem.

#### 5.C.ii. The curves at two signal-to-noise ratios

For $r^2/\sigma^2=5$ and for $r^2/\sigma^2=0.5$, use 5.C.i.1 to locate the
min-norm risk's minimum on $\gamma>1$, if it has one, and say whether the curve
crosses the null risk $r^2$. Then sketch both curves against $\gamma$ on a log
axis, with $r^2$ as a horizontal line, the bias and variance contributions
shaded, and optimal ridge on the same axes. The interactive figure below draws
all of these.

*What a correct sketch shows.* For both SNRs, the curve starts at $0$, rises as $\sigma^2\gamma/(1-\gamma)$, and has a vertical asymptote at $\gamma=1$, entirely from variance; the bias is zero for $\gamma<1$.
- For SNR $5$ ($r\approx2.24\sigma$): past the peak the curve falls *through* $r^2$, has its minimum at $\gamma=r/(r-\sigma)\approx1.81$ with value $2r\sigma-\sigma^2\approx3.47\sigma^2<r^2=5\sigma^2$, and then climbs back to $r^2$ from below.
- For SNR $0.5$: past the peak the curve falls toward $r^2$ from above and never crosses it.

Optimal ridge is a smooth, monotone, non-decreasing curve that lies below min-norm everywhere and tends to $r^2$ as $\gamma\to\infty$.

▶ **Interactive figure:** [`double_descent.html`](assets/interactive/double_descent.html). Turn on the bias/variance decomposition and check which term makes the peak. Toggle optimal ridge λ* = σ²γ/r². Drag the SNR across 1 and watch whether the min-norm curve dips below the null risk r². The dots are finite-n simulations: note how they scatter near γ = 1.

### 5.D. What regime are we in?

Classical asymptotics fix $p$ and send $n\to\infty$. §5 lives at $p/n=O(1)$.
[Chinchilla](https://arxiv.org/abs/2203.15556)'s compute-optimal allocation, of
roughly 20 training tokens per parameter, puts LLM pretraining at
$p/n\approx1/20$ if a token counted as a sample: neither $p\ll n$ nor
$p\gg n$. Two cautions. Tokens are not i.i.d. regression samples. And
pretraining usually runs for about one epoch, so the training loss is never
driven to zero, and nothing interpolates. Networks show double-descent-like
curves in width, training time and sample size
([Nakkiran et al. 2019](https://arxiv.org/abs/1912.02292)), typically with
label noise, and these are weakened by regularization and early stopping. That
phenomenology is consistent with the variance mechanism derived here, but it is
not derived from it.

#### 5.D.i. Question

Using §3.A, describe a measurement that would tell you whether a network's
double-descent peak is a variance divergence of the kind derived here.

Train several networks at the same size on independent data draws (or with different seeds and different label noise) near the apparent peak. Decompose the test loss into the loss of the ensemble average (bias plus noise) and the spread across the ensemble (variance), as in §3.A. If the peak is a variance divergence, the spread peaks there while the ensemble-average loss does not. Weight decay tuned per size should then flatten the peak. A second check is the spectrum of the relevant kernel or feature covariance at the peak: there should be near-zero eigenvalues.

### 5.E. Reading (optional)

**[Surprises in high-dimensional ridgeless least squares interpolation](https://arxiv.org/abs/1903.08560)**, Trevor Hastie, Andrea Montanari, Saharon Rosset, Ryan J. Tibshirani (Annals of Statistics, 2022) · *optional*

Exact asymptotics, as p/n → γ, for the risk of minimum-norm interpolation and ridge, under isotropic and general covariance, in well-specified and misspecified linear models and a random-features model. Double descent appears in the simplest of these, the peak is a variance divergence, and optimally tuned ridge has no peak. Read §1 (overview and figures) and §3 (isotropic features: Theorem 1 is the formula you derived in 5.B, for general i.i.d. entries rather than only Gaussian ones). §5 on misspecified models shows when the global minimum can sit at γ > 1. §4 (general covariance) and §8 (nonlinear random features) are for those heading to WB5.

<a id="wb2-section-6"></a>

## 6. Information bottleneck and the information plane

**The result.** For a deterministic network, $I(X;T)=H(T)$ when the input is discrete;
it equals $H(X)$ while the layer is injective on the input support. If the
layer output has an atomless distribution, the mutual information is infinite. The "compression phase" seen in information-plane
plots of an injective deterministic layer therefore reflects the estimator. A binned estimate is the
entropy of a quantization, and saturating units shrink it. Once noise is added
to the layer, $I(X;T)$ becomes finite, and in the two-class model below it falls
when within-class covariance shrinks at fixed class means and noise scale.
6.F.i asks what these derivations do and do not settle about the published
accounts.

**Notation for this section.** $X$ is the input, $Y$ the label, and
$T=f(X)$ the activations of one hidden layer. $T$ is random because $X$ is.
$H(\cdot)$ is the entropy of a discrete variable and $h(\cdot)$ the
differential entropy of a density, both in nats. Mutual information is
$I(A;B)=H(A)-H(A\mid B)$, or the same with $h$. Because $T$ depends on $Y$ only
through $X$, the variables form a Markov chain $Y\to X\to T$, and the data
processing inequality gives $I(T;Y)\le I(X;Y)$.

### 6.A. Three accounts of the information plane

The *information bottleneck* (IB), introduced by
[Tishby, Pereira and Bialek (1999)](https://arxiv.org/abs/physics/0004057),
asks for a representation $T$ of $X$ that is as compressed as possible while
staying informative about $Y$:

$$
\min_{p(t\mid x)}\ I(X;T)-\beta\,I(T;Y),\qquad \beta>0 .
$$

The *information plane* plots each layer of a network as the point
$\big(I(X;T),\,I(T;Y)\big)$ and follows it through training.

**Tishby's claim** (Shwartz-Ziv and Tishby 2017). Training has two phases. In
a short *fitting* phase both coordinates rise. In a long *compression* phase
$I(X;T)$ falls while $I(T;Y)$ holds. The compression is driven by the
noise-dominated, diffusive part of SGD, and it is why networks generalize.
Their estimator binned the activations of a $\tanh$ network into 30 equal bins
on $[-1,1]$, with $X$ uniform over 4096 inputs.

**The critique** (Saxe et al. 2018). Compression appears with double-sided
saturating units like $\tanh$, and not with ReLU or linear units. Networks that
do not compress still generalize. The same curves appear under full-batch
gradient descent, so SGD noise is not their cause. There is also a conceptual
problem, which 6.B makes precise: for a deterministic network the quantity
being plotted is constant or infinite.

**A third reading** (Goldfeld et al. 2019). Add a
little Gaussian noise to each layer, so that $I(X;T)$ is finite and depends on
the weights. Then compression does occur, and it is driven by the clustering
of same-class representations. The binned estimates track the same clustering.

### 6.B. A deterministic network has nothing to compress

#### 6.B.i. Constant or infinite

1. Let $X$ be discrete and $T=f(X)$ for a deterministic $f$. Show that $I(X;T)=H(T)$, and that $I(X;T)=H(X)$ if $f$ is injective on the support of $X$.
2. In Shwartz-Ziv and Tishby's setup, $X$ is uniform over 4096 inputs. What is $I(X;T)$ for any layer that maps distinct inputs to distinct activation vectors? With 32-bit floats, can two distinct inputs still be mapped to exactly the same activation vector, and what would cause it?
3. Now let $X$ have a density on $\mathbb R^d$, and suppose $T=f(X)$ has an atomless distribution, i.e. $\Pr[T=t]=0$ for every $t$ (true, for example, if $f$ is smooth and injective). Show that $I(X;T)=\infty$.
4. Conclude: which changes to the weights can change $I(X;T)$ in cases 1 and 3?

<details><summary>Hint</summary>

For 3, let $q_\Delta$ quantize $T$ to cubes of side $\Delta$. Since $q_\Delta(T)$ is a function of $X$, part 1 and data processing give $I(X;T)\ge I(X;q_\Delta(T))=H(q_\Delta(T))$. Then use $H(Q)\ge-\log\max_k\Pr[Q=k]$.
</details>

**1.** $I(X;T)=H(T)-H(T\mid X)$, and $H(T\mid X)=0$ because $T$ is determined by $X$. So $I(X;T)=H(T)=H(f(X))$. If $f$ is injective on the support, $f(X)$ is a relabelling of $X$, so $H(f(X))=H(X)$.

**2.** $H(X)=\log4096=12$ bits $\approx8.32$ nats, for every such layer, at every step of training. This value remains constant while the mapping stays injective. Float precision alone does not determine collision probabilities: dead ReLUs, finite-precision saturation, and other noninjective maps can produce exact collisions. Changes in a binned estimate need not reflect changes in this unquantized information.

**3.** Let $q_\Delta$ map $T$ to the cube of side $\Delta$ that contains it. $q_\Delta(T)$ is a deterministic function of $X$, so by 1, $I(X;q_\Delta(T))=H(q_\Delta(T))$. Since $X\to T\to q_\Delta(T)$, data processing gives $I(X;T)\ge H(q_\Delta(T))\ge-\log\max_k\Pr[q_\Delta(T)=k]$.

It remains to show that $m(\Delta):=\sup_{\text{cubes }Q\text{ of side }\Delta}\Pr[T\in Q]\to0$. Suppose not: then there are $\epsilon>0$ and cubes $Q_j$ with sides $\Delta_j\to0$ and $\Pr[T\in Q_j]\ge\epsilon$. By tightness, the $Q_j$ eventually lie in a compact set, so a subsequence of their centres converges to some $t_0$. Every ball around $t_0$ then has probability at least $\epsilon$, so $\Pr[T=t_0]\ge\epsilon$, contradicting atomlessness. Hence $H(q_\Delta(T))\to\infty$ and $I(X;T)=\infty$.

(Equivalently, when $T$ has a density, $h(T\mid X)=-\infty$ because $T$ given $X$ is a point mass.)

**4.** In case 3, none: $I(X;T)=\infty$ for every weight setting that keeps $T$ atomless. In case 1, only exact collisions, where two distinct inputs are mapped to exactly the same activation vector, can lower $H(f(X))$. Such collisions are possible, for example with dead ReLUs. The constancy conclusion requires injectivity on the discrete input support; the infinite-information conclusion requires atomlessness. Under those conditions, movement in a finite plotted estimate does not describe a change in the exact mutual information.

### 6.C. What the binned estimator measures

The binned estimator replaces $T$ by $\hat T=\mathrm{bin}(T)$, its bin index,
and reports $I(X;\hat T)$. Since $\hat T$ is a function of $X$, 6.B.i.1 says
this is just $H(\mathrm{bin}(T))$, the entropy of a quantization.

#### 6.C.i. Saturation looks like compression

Take one unit with standard normal pre-activation $Z$ and pre-activation scale
$a>0$, the weight norm. Training typically grows $a$.

1. $T=\tanh(aZ)$, binned into $K$ equal bins on $[-1,1]$, with $K$ even. Show that $H(\mathrm{bin}(T))\to\log2$ as $a\to\infty$. What is the limit as $a\to0$? How does it change if $K$ is odd?
2. Sketch $H(\mathrm{bin}(T))$ as a function of $a$ for $K=30$. Explain why steadily growing weights alone produce a "fitting phase" followed by a "compression phase".
3. $T=\mathrm{ReLU}(aZ)=\max(0,aZ)$, binned with a fixed width $\Delta$: $[0,\Delta),[\Delta,2\Delta),\dots$. Show that as $a\to\infty$,
$$
H(\mathrm{bin}(T))=\log2+\tfrac12\Big(h(|Z|)+\log\frac a\Delta\Big)+o(1),\qquad h(|Z|)=\tfrac12\log\frac{\pi e}2 .
$$
So the estimate grows without bound, like $\tfrac12\log a$.
4. Many implementations instead bin ReLU activations into $K$ equal bins on $[0,\max T]$, the observed range. What happens to the estimate as $a$ grows?
5. In one sentence: what quantity is the binned estimator measuring?

<details><summary>Hint</summary>

For 1, the bin edges are $e_j=-1+2j/K$; compute $\Pr[T\ge1-2/K]$. For 3, condition on the sign of $Z$, and use the standard fact that for a nice density, $H(\mathrm{bin}_\delta(W))=h(W)-\log\delta+o(1)$ as $\delta\to0$. Here the effective bin width for $Z$ is $\delta=\Delta/a$. For 4, ReLU is positively homogeneous.
</details>

**1.** The top bin is $[1-2/K,1]$, and
$\Pr[T\ge1-2/K]=\Pr\big[Z\ge\operatorname{artanh}(1-2/K)/a\big]\to\Pr[Z>0]=\tfrac12$
as $a\to\infty$. The bottom bin likewise tends to $\tfrac12$, so every other
bin tends to $0$. Entropy is continuous on the finite simplex, so
$H\to\log2$: the unit has become a sign function, and the estimate counts
two cells.

As $a\to0$, $T\approx aZ$ concentrates at $0$. With $K$ even, $0$ is a bin
edge, so the two bins adjacent to $0$ each get mass $\to\tfrac12$ and
$H\to\log2$ again. With $K$ odd, the middle bin straddles $0$, takes all the
mass, and $H\to0$.

**2.** For $K=30$, exact bin probabilities give (from `checks/wb2b_check.py`):

| $a$ | 0.1 | 0.5 | 0.9 | 2 | 5 | 20 | 100 | $10^4$ |
|---|---|---|---|---|---|---|---|---|
| $H(\mathrm{bin}\tanh(aZ))$ (nats) | 1.83 | 3.21 | 3.40 (max) | 2.88 | 1.92 | 1.10 | 0.80 | 0.69 |

At small $a$ the activations occupy a few central bins. As $a$ grows they
spread over more bins, so the estimate rises ("fitting"). Past
$a\approx1$ they pile up in the two saturated end bins, so it falls toward
$\log2$ ("compression"). Monotone weight growth produces the two-phase curve
with no change in what the network distinguishes: $T$ stays an injective
function of $Z$ throughout.

**3.** Let $S=\mathbf 1[Z>0]$. All mass with $Z\le0$ sits at $T=0$, in bin
$0$, together with $\Pr[0<Z<\Delta/a]\to0$. Up to that vanishing overlap,
$S$ is a function of the bin, so
$$
H(\mathrm{bin}\,T)=H(S)+\Pr[S=1]\,H(\mathrm{bin}\,T\mid S=1)+o(1)
=\log2+\tfrac12H\big(\mathrm{bin}_{\Delta/a}(|Z|)\big)+o(1).
$$
Given $Z>0$, $Z$ has the density of $|Z|$, and binning $aZ$ at width $\Delta$
is binning $Z$ at width $\delta=\Delta/a\to0$. So
$H(\mathrm{bin}_\delta|Z|)=h(|Z|)-\log\delta+o(1)$. The half-normal has
$h(|Z|)=\tfrac12\log(2\pi e)-\log2=\tfrac12\log(\pi e/2)$, which gives the
formula. Check: with $\Delta=1/15$ and $a=100$ the formula gives $4.713$
nats and the exact value is $4.710$; at $a=1000$, $5.864$ against $5.864$.
The ReLU estimate keeps growing, with no compression phase. This is the
pattern Saxe et al. report.

**4.** $\max(0,aZ)/\max_i(0,aZ_i)$ does not depend on $a$, so the binned
estimate is exactly constant in $a$. The answer to 3 versus 4 is decided by a
choice of bins, not by the network.

**5.** It measures how many cells of size $\Delta$ the data occupy, i.e. the
spread of the representation measured at a resolution fixed by the analyst.
That is a geometric quantity. For a fixed binning rule it is a property of the distribution of $T$,
but it is not generally equal to the unquantized $I(X;T)$. The distribution
and geometry of $T$ can change even while the latter stays constant.

### 6.D. With noise, compression is clustering

Goldfeld et al. make the mutual information meaningful by adding noise:
$T=f(X)+\sigma\varepsilon$ with $\varepsilon\sim\mathcal N(0,I_d)$ independent
of $X$. Now $h(T\mid X)=h(\sigma\varepsilon)=\frac d2\log(2\pi e\sigma^2)$ is
finite, and $I(X;T)=h(T)-\frac d2\log(2\pi e\sigma^2)$ depends on $f$.
Differential entropy and the Gaussian channel are in Cover and Thomas,
*Elements of Information Theory*, ch. 8–9.

#### 6.D.i. The Gaussian channel

1. Suppose $f(X)\sim\mathcal N(\mu,S)$. Show that $I(X;T)=\tfrac12\log\det(I+S/\sigma^2)=\sum_k\tfrac12\log(1+s_k/\sigma^2)$, where $s_k$ are the eigenvalues of $S$.
2. Show that for any $f(X)$ with covariance $S$, $I(X;T)\le\tfrac12\log\det(I+S/\sigma^2)$.
3. Interpret $\tfrac12\log(1+s_k/\sigma^2)$ in terms of the number of distinguishable cells of size $\sigma$ along the $k$-th principal direction. Compare with 6.C.i.5.

<details><summary>Hint</summary>

$h(T\mid X)=h(\sigma\varepsilon)$ is the same for every $f$. For 2, among all densities with a given covariance, the Gaussian has the largest differential entropy.
</details>

#### 6.D.ii. A two-class model

Let $Y\in\{\pm1\}$ be uniform, and let $X=(Y,U)$ with $U\sim\mathcal N(0,I_d)$
independent of $Y$, so the label is a function of the input. The layer computes
$$
f(X)=Y\,m+W^{1/2}U ,
$$
so $m\in\mathbb R^d$ sets the class means $\pm m$ (between-class covariance
$mm^\top$) and $W$ is the within-class covariance. Then $T=f(X)+\sigma\varepsilon$.

1. Show that $I(X;T)=I(T;Y)+I(X;T\mid Y)$, and that $I(X;T\mid Y)=\tfrac12\log\det(I+W/\sigma^2)$.
2. Let $\Sigma_T=W+\sigma^2I$ and $d^2=m^\top\Sigma_T^{-1}m$. Show that $I(T;Y)$ depends on the model only through $d^2$, and that
$$
I(T;Y)=\log2-\mathbb E\log\!\big(1+e^{-L}\big),\qquad L\sim\mathcal N(2d^2,\,4d^2).
$$
Show that this increases with $d^2$ and tends to $\log2$.
3. Now shrink the within-class covariance, $W\to\alpha W$ with $\alpha\downarrow0$, keeping $m$ and $\sigma$ fixed. Describe the path of the point $\big(I(X;T),I(T;Y)\big)$. What are its limits?
4. What happens to $I(X;T)$ if you rescale the whole representation, $f\to cf$, and leave $\sigma$ fixed? What does this say about the role of $\sigma$?

<details><summary>Hint</summary>

For 1, use the chain rule on $I(X,Y;T)$ and the fact that $Y$ is a function of $X$. Given $Y$, $f(X)$ is Gaussian, so 6.D.i.1 applies. For 2, the posterior log-odds $L=\log\frac{p(T\mid Y=+1)}{p(T\mid Y=-1)}$ is linear in $T$. For monotonicity, show that a channel with smaller $d^2$ is the channel with larger $d^2$ followed by extra noise.
</details>

**6.D.i.1.** $T\sim\mathcal N(\mu,S+\sigma^2I)$, so
$h(T)=\tfrac12\log\det\big(2\pi e(S+\sigma^2I)\big)$ and
$h(T\mid X)=\tfrac12\log\det(2\pi e\sigma^2I)$. Subtracting gives
$\tfrac12\log\det(I+S/\sigma^2)$. Diagonalizing $S$ gives the sum.

**6.D.i.2.** $h(T\mid X)$ is the same for every $f$. $T$ has covariance
$S+\sigma^2I$, and the Gaussian maximizes differential entropy at fixed
covariance, so $h(T)\le\tfrac12\log\det(2\pi e(S+\sigma^2I))$.

**6.D.i.3.** $\tfrac12\log(1+s_k/\sigma^2)\approx\log(\sqrt{s_k}/\sigma)$ when
$s_k\gg\sigma^2$. This is the log of the number of noise-width cells that the
signal's spread covers in that direction. The noisy mutual information counts
cells of size $\sigma$, exactly as the binned estimate counts cells of size
$\Delta$. In both, the analyst's choice of scale is what makes the number
finite.

**6.D.ii.1.** $Y$ is a function of $X$, so $I(X;T)=I(X,Y;T)=I(Y;T)+I(X;T\mid Y)$
by the chain rule. Given $Y=y$, $f(X)=ym+W^{1/2}U\sim\mathcal N(ym,W)$, so
by 6.D.i.1, $I(X;T\mid Y=y)=\tfrac12\log\det(I+W/\sigma^2)$ for both values of
$y$.

**6.D.ii.2.** Given $Y=y$, $T\sim\mathcal N(ym,\Sigma_T)$. The log-likelihood
ratio is
$L=\log\frac{p(T\mid+)}{p(T\mid-)}=2m^\top\Sigma_T^{-1}T$. Given $Y=+1$, $L$ has
mean $2m^\top\Sigma_T^{-1}m=2d^2$ and variance
$4m^\top\Sigma_T^{-1}\Sigma_T\Sigma_T^{-1}m=4d^2$. The posterior is
$\Pr[Y=+1\mid T]=1/(1+e^{-L})$, so given $Y=+1$, $-\log\Pr[Y\mid T]=\log(1+e^{-L})$,
and the case $Y=-1$ is the mirror image. Hence
$I(T;Y)=H(Y)-H(Y\mid T)=\log2-\mathbb E\log(1+e^{-L})$, which depends only on
$d^2$. (Since $L$ is a sufficient statistic, $I(T;Y)=I(L;Y)$.)

Monotonicity: $L/(2d)=Yd+G$ with $G\sim\mathcal N(0,1)$. For $d'<d$,
$Yd'+G'\overset{\text{law}}{=}\frac{d'}{d}(Yd+G)+\sqrt{1-d'^2/d^2}\,G''$, so
the $d'$ channel is the $d$ channel followed by independent noise. By data
processing, $I$ is non-decreasing in $d$. As $d\to\infty$, $L\to+\infty$ given
$Y=+1$, so $I\to\log2=H(Y)$.

**6.D.ii.3.** With $W\to\alpha W$:
$I(X;T\mid Y)=\tfrac12\log\det(I+\alpha W/\sigma^2)\downarrow0$, while
$d^2=m^\top(\alpha W+\sigma^2I)^{-1}m\uparrow\|m\|^2/\sigma^2$, so $I(T;Y)$
is nondecreasing. The point moves **left and up** (weakly: a coordinate can
stay fixed in degenerate cases such as $W=0$). To justify the leftward movement, for $\alpha_2>\alpha_1$
the marginal law of $T_{\alpha_2}$ is that of $T_{\alpha_1}$ plus independent
Gaussian noise with covariance $(\alpha_2-\alpha_1)W$. Adding independent
noise cannot decrease differential entropy here, while $h(T\mid X)$ stays
fixed. Thus $I(X;T)$ is nondecreasing in $\alpha$. In the limit $I(X;T)=I(T;Y)$:
the layer carries no information about $X$ beyond what it carries about $Y$. A numerical example from `checks/wb2b_check.py`, with $m=e_1$,
$W=\mathrm{diag}(0.6,1,1)$, $\sigma=0.5$ (nats; Monte Carlo agrees with the
formulas to $\le0.004$):

| $\alpha$ | 1 | 0.3 | 0.1 | 0.01 |
|---|---|---|---|---|
| $I(X;T)$ | 2.59 | 1.59 | 1.04 | 0.68 |
| $I(T;Y)$ | 0.37 | 0.53 | 0.60 | 0.63 |

In this model, compression in the noisy-network sense *is* within-class
clustering. The conditional information
$I(X;T\mid Y)=\tfrac12\log\det(I+W/\sigma^2)$ measures within-class
spread in units of $\sigma$. The drop in total $I(X;T)$ is the drop in
this conditional term **minus** the gain in label information $I(T;Y)$;
it is not equal to the conditional term alone.

**6.D.ii.4.** $f\to cf$ multiplies $W$ by $c^2$ and $m$ by $c$, which is the
same as $\sigma\to\sigma/c$. $I(X;T)$ grows without bound as $c\to\infty$ if $W$ has a positive
eigenvalue. If $W=0$, the signal contains only the binary label and the
information is bounded by $\log2$ (approaching it if $m\ne0$).
The noisy mutual information is therefore not a property of the
representation's shape alone but of its size relative to $\sigma$, the scale
the analyst injected. Read with 6.C, both estimators measure geometry at a
chosen resolution. Whether that makes mutual information the wrong quantity,
or the right one at a stated scale, is not settled by these calculations.

### 6.E. The IB as rate–distortion (optional)

Rate–distortion theory asks for the encoder $p(t\mid x)$ that minimizes the
rate $I(X;T)$ subject to an expected distortion
$\mathbb E\,d(X,T)\le D$ (Cover and Thomas, *Elements of Information Theory*,
ch. 10). Part 3 is the self-consistent equation of Tishby, Pereira and Bialek
(linked in 6.A).

#### 6.E.i. The distortion hidden in the IB

1. Define $p(y\mid t)=\sum_xp(y\mid x)\,p(x\mid t)$ and $d(x,t)=\mathrm{KL}\big(p(y\mid x)\,\|\,p(y\mid t)\big)$. Show that $\mathbb E\,d(X,T)=I(X;Y)-I(T;Y)$, so the IB Lagrangian is the rate–distortion Lagrangian $I(X;T)+\beta\,\mathbb E\,d(X,T)$ up to a constant.
2. Why is this not an ordinary rate–distortion problem?
3. Show that at a stationary point, $p(t\mid x)=\dfrac{p(t)}{Z(x,\beta)}\exp\!\big(-\beta\,\mathrm{KL}(p(y\mid x)\,\|\,p(y\mid t))\big)$.
4. Seen as a regularizer on a network, what does penalizing $I(X;T)$ in the noisy-channel model of 6.D penalize?

<details><summary>Hint</summary>

For 3, differentiate $I(X;T)-\beta I(T;Y)$ with respect to $p(t\mid x)$, with a multiplier $\lambda(x)$ for $\sum_tp(t\mid x)=1$. Terms that depend only on $x$ can be absorbed into $\lambda(x)$.
</details>

**1.** With $Y\to X\to T$, the joint is $p(x)p(y\mid x)p(t\mid x)$. Then
$$
\mathbb E\,d=\sum_{x,t,y}p(x,t)\,p(y\mid x)\log\frac{p(y\mid x)}{p(y\mid t)}
=-H(Y\mid X)+H(Y\mid T)=I(X;Y)-I(T;Y),
$$
using $p(x,t)p(y\mid x)=p(x,y,t)$. Hence
$I(X;T)-\beta I(T;Y)=I(X;T)+\beta\,\mathbb E\,d-\beta I(X;Y)$, and the last
term does not depend on the encoder.

**2.** The distortion depends on the encoder itself, through $p(y\mid t)$.
In rate–distortion theory $d$ is fixed in advance. This is why the IB
equations must be solved self-consistently, by iterating.

**3.** $\partial I(X;T)/\partial p(t\mid x)=p(x)\big[\log\frac{p(t\mid x)}{p(t)}\big]+(\text{terms in }x\text{ only})$.
For $I(T;Y)$, with $p(t,y)=\sum_xp(t\mid x)p(x,y)$ and $p(t)=\sum_xp(t\mid x)p(x)$,
$$
\frac{\partial I(T;Y)}{\partial p(t\mid x)}=\sum_yp(x,y)\log\frac{p(y\mid t)}{p(y)}
=p(x)\Big[-\mathrm{KL}\big(p(y\mid x)\|p(y\mid t)\big)+\sum_yp(y\mid x)\log\frac{p(y\mid x)}{p(y)}\Big],
$$
where the terms from differentiating inside the logarithm cancel. The last
sum depends only on $x$. Setting the derivative of the Lagrangian to zero
gives $\log\frac{p(t\mid x)}{p(t)}+\beta\,\mathrm{KL}(\cdot\|\cdot)=-\tilde\lambda(x)$,
which is the stated form with $Z(x,\beta)$ the normalizer.

**4.** By 6.D.i.2, $I(X;T)\le\tfrac12\log\det(I+S/\sigma^2)$, and 6.D.ii.1
splits it into a label term and $\tfrac12\log\det(I+W/\sigma^2)$. Penalizing
it pushes the representation to have small spread in units of $\sigma$, and,
at fixed label information, small within-class spread. It acts as a penalty on
activation scale and on within-class variance: a clustering pressure, once
more a geometric statement.

### 6.F. Readings

**[Opening the black box of deep neural networks via information](https://arxiv.org/abs/1703.00810)**, Ravid Shwartz-Ziv, Naftali Tishby (arXiv, 2017) · *core*

The fitting-then-compression account summarized in 6.A, with its information-plane figures; it also argues that compression explains the benefit of depth. Read the IB setup and the description of the binning estimator (30 bins, 4096 inputs), then the information-plane figures and the drift-then-diffusion account of SGD. Read it against 6.B and 6.C: at each figure, ask what the plotted quantity is for a deterministic network.

**[On the information bottleneck theory of deep learning](https://openreview.net/forum?id=ry_WPG-A-)**, Andrew M. Saxe, Yamini Bansal, Joel Dapello, Madhu Advani, Artemy Kolchinsky, Brendan D. Tracey, David D. Cox (ICLR 2018) · *core*

The critique summarized in 6.A, with experiments comparing tanh, ReLU and linear networks and full-batch versus stochastic training. Read the introduction and the sections on the nonlinearity (the binning analysis is the one you did in 6.C), on compression and generalization, and on batch vs stochastic training. The linear-network analysis, where task-irrelevant information can be compressed while fitting, is optional.

**[Estimating information flow in deep neural networks](https://arxiv.org/abs/1810.05728)**, Ziv Goldfeld, Ewout van den Berg, Kristjan Greenewald, Igor Melnyk, Nam Nguyen, Brian Kingsbury, Yury Polyanskiy (ICML 2019) · *core*

The noisy-network reading summarized in 6.A, with a rigorous estimator for I(X;T) in noisy networks and small worked examples of compression as clustering. Read the introduction and the setup for why deterministic I(X;T) is degenerate and how noise fixes it (6.B and 6.D.i), then the section relating compression to clustering, with its small worked examples. The estimator's convergence theory can be skipped.

#### 6.F.i. What the derivations settle

The three accounts in 6.A amount to three positions: (i) compression in mutual
information is real and causes generalization; (ii) it is an artefact of
saturation and binning, not needed for generalization; (iii) it is real
within-class clustering, which binned estimates also track. What does each of
6.B, 6.C and 6.D establish or rule out about these positions? Which question
does none of them address, and what kind of experiment would?

- **6.B** rules out the literal reading of (i) for deterministic networks:
  noise-free $I(X;T)$ is constant while the layer stays injective on a
  discrete input set (it can fall only through exact collisions), and
  infinite when the representation is atomless. So "compression in mutual
  information" must mean compression of some estimate, or of a noisy
  surrogate.
- **6.C** supports (ii)'s mechanism. Saturation alone produces rise-then-fall
  in the tanh estimate, and the ReLU estimate grows or stays constant
  depending on the bins. But it also supports (iii): the estimate is a
  spread-at-resolution-$\Delta$ statistic, which is geometric, so "artefact"
  and "measures geometry" are both true.
- **6.D** shows that once noise defines a scale, the drop in $I(X;T)$ at
  fixed label information is exactly within-class shrinkage. That is (iii)'s
  claim, made exact in a toy model. Whether mutual information is the right
  name for it is now a question about invariances (6.D.ii.4). Noise-free
  mutual information is unchanged by any invertible map of $T$; a geometric
  statistic, such as the ratio of within-class to between-class scatter, is
  not. Two representations with equal estimated $I(X;T)$ and $I(T;Y)$ but
  different geometry would separate the two.
- None of 6.B–6.D bears on causation (does clustering *cause*
  generalization?). That needs interventions: for example, penalize any
  decrease in within-class spread and see whether test loss suffers. Two
  observational checks are already in Saxe et al.: ReLU networks generalize
  without compressing in the binned estimate, and the compression phase also
  appears under full-batch training, which rules out SGD noise as its cause.

<a id="wb2-section-7"></a>

## 7. Bayes and MLE generalization error

**The result.** In the regular regime (fixed $p$, $n\to\infty$, a nondegenerate
minimum), the ERM parameter fluctuates with the sandwich covariance
$H^{-1}CH^{-1}/n$. Its expected excess risk is $\mathrm{Tr}(H^{-1}C)/2n$, and
its expected optimism (test minus training loss) is twice that. When the
model is a well-specified likelihood, $C=H$ and these become $p/2n$ and
$p/n$: AIC. The Laplace approximation to the evidence gives BIC's
$\frac p2\log n$. The optional 7.E compares a single fit with the Bayes
ensemble and with a single posterior draw: in the regular case their errors
are $p/2n$, $p/2n$ and $p/n$, and in singular models the real log canonical
threshold replaces $p/2$ in the Bayes error.

*Sources:* the sandwich is due to Huber (1967) and, for misspecified
likelihoods, H. White, "Maximum likelihood estimation of misspecified models",
*Econometrica* (1982). AIC is from Akaike (1974) and BIC from
[Schwarz (1978)](https://doi.org/10.1214/aos/1176344136); ESL §7.4–7.5 and
§7.7 derive both in textbook form.

**Setup, self-contained.** Data $z=(x,y)\sim P$, sample of size $n$, loss
$\ell(\theta;z)$, population risk $R(\theta)=\mathbb E\,\ell(\theta;z)$,
empirical risk $\hat R_n(\theta)=\frac1n\sum_i\ell(\theta;z_i)$, ERM
$\hat\theta_n$, population minimizer $\theta^*$. Assume:

- $p$ is fixed and $n\to\infty$; $\theta^*$ is the unique minimizer, in the interior;
- $H:=\nabla^2R(\theta^*)\succ0$, and $\hat\theta_n\to\theta^*$ in probability;
- enough smoothness and moments for Taylor expansions, the law of large numbers and the central limit theorem.

Define the gradient-noise covariance at the optimum,
$$
C:=\mathrm{Cov}\big[\nabla_\theta\ell(\theta^*;z)\big]=\mathbb E\big[\nabla\ell(\theta^*;z)\nabla\ell(\theta^*;z)^\top\big],
$$
where the two agree because $\mathbb E\nabla\ell(\theta^*;z)=\nabla R(\theta^*)=0$.

From §4, restated: if $\theta=\theta^*+\delta$ with $\mathbb E\,\delta\delta^\top=\Sigma$,
then $\mathbb E\,R(\theta)-R(\theta^*)\approx\tfrac12\mathrm{Tr}(H\Sigma)$,
because $\nabla R(\theta^*)=0$.

**Loss conventions.** All excess risks below are in loss units. For squared
loss this section uses $\ell=\tfrac12(y-f_\theta(x))^2$, so the Gauss–Newton term has no factor of 2; the full Hessian also has
a residual-times-second-derivative term, whose population expectation
vanishes for a correctly specified conditional mean. (§1 used $(y-f)^2$, which doubles $H$
and quadruples $C$; $\mathrm{Tr}(H^{-1}C)$ doubles.) For the log loss
$\ell=-\log p_\theta(y\mid x)$, in nats, and a well-specified model,
$R(\theta)-R(\theta^*)=\mathbb E_x\,\mathrm{KL}\big(p_{\theta^*}(\cdot\mid x)\,\|\,p_\theta(\cdot\mid x)\big)$,
so excess risk is the KL divergence to the truth.

### 7.A. The sandwich

#### 7.A.i. Asymptotic normality of ERM

Starting from the first-order condition $\nabla\hat R_n(\hat\theta_n)=0$, show
that
$$
\sqrt n\,(\hat\theta_n-\theta^*)\ \Rightarrow\ \mathcal N\big(0,\ H^{-1}CH^{-1}\big).
$$
Which of $H$ and $C$ would a "Fisher information" formula use, and when does
that give the wrong answer?

<details><summary>Hint</summary>

Taylor-expand $\nabla\hat R_n$ around $\theta^*$. Apply the law of large numbers to $\nabla^2\hat R_n(\theta^*)$ and the central limit theorem to $\sqrt n\,\nabla\hat R_n(\theta^*)$, then Slutsky.
</details>

Expanding,
$0=\nabla\hat R_n(\hat\theta_n)=\nabla\hat R_n(\theta^*)+\nabla^2\hat R_n(\tilde\theta)\,(\hat\theta_n-\theta^*)$
for some $\tilde\theta$ between $\hat\theta_n$ and $\theta^*$ (row by row). So
$$
\sqrt n(\hat\theta_n-\theta^*)=-\big[\nabla^2\hat R_n(\tilde\theta)\big]^{-1}\sqrt n\,\nabla\hat R_n(\theta^*).
$$
$\nabla\hat R_n(\theta^*)=\frac1n\sum_i\nabla\ell(\theta^*;z_i)$ is an average
of i.i.d. vectors with mean $0$ and covariance $C$, so by the CLT
$\sqrt n\,\nabla\hat R_n(\theta^*)\Rightarrow\mathcal N(0,C)$. By the LLN and
consistency, $\nabla^2\hat R_n(\tilde\theta)\to H$ in probability. By
Slutsky, the limit is $-H^{-1}\mathcal N(0,C)=\mathcal N(0,H^{-1}CH^{-1})$.
This is the *sandwich*, with $H^{-1}$ as the bread and $C$ as the filling
(Huber; White 1982 for misspecified likelihoods).

The "inverse Fisher information" formula is $H^{-1}$ (or $C^{-1}$), which
equals the sandwich only when $C=H$ (7.C). Misspecification can break this equality, although it need not do so
in every model. Example: heteroscedastic noise fitted with a homoscedastic
model, as in 7.B.ii.

### 7.B. Excess risk, optimism, and a factor of two

#### 7.B.i. Two traces

1. Show that $\mathbb E\,R(\hat\theta_n)-R(\theta^*)\approx\frac1{2n}\mathrm{Tr}(H^{-1}C)$.
2. Show that the expected optimism is twice as large:
$$
\mathbb E\big[R(\hat\theta_n)-\hat R_n(\hat\theta_n)\big]\approx\frac1n\mathrm{Tr}(H^{-1}C).
$$
This is Takeuchi's information criterion (TIC): estimate the test loss by $\hat R_n(\hat\theta_n)+\frac1n\mathrm{Tr}(\hat H^{-1}\hat C)$.
3. Explain the factor of two in one sentence. Which of the two quantities does a held-out set estimate?

<details><summary>Hint</summary>

For 2, expand $\hat R_n$ around its own minimizer $\hat\theta_n$ and evaluate the expansion at $\theta^*$. Then use $\mathbb E\hat R_n(\theta^*)=R(\theta^*)$.
</details>

#### 7.B.ii. A concrete misspecified case

Let $x\sim\mathcal N(0,1)$ and $y=\beta x+\varepsilon$, where
$\varepsilon\mid x\sim\mathcal N(0,\tau^2x^2)$. The noise is heteroscedastic.
Fit $y\approx\theta x$ with $\ell=\tfrac12(y-\theta x)^2$.

1. Compute $H$, $C$, the asymptotic variance of $\hat\theta_n$, the expected excess risk and the expected optimism.
2. Compare each with what you get by assuming homoscedastic noise of the same average variance, $\mathbb E\varepsilon^2=\tau^2$, i.e. with $C=\tau^2H$. By what factor is the naive answer wrong?

<details><summary>Hint</summary>

At $\theta^*=\beta$ the per-sample gradient is $-\varepsilon x$. For a standard normal, $\mathbb E x^4=3$.
</details>

**7.B.i.1.** By 7.A, $\delta=\hat\theta_n-\theta^*$ has
$\mathbb E\,\delta\delta^\top\approx\Sigma_n:=H^{-1}CH^{-1}/n$. By the §4
formula, the expected excess risk is
$\tfrac12\mathrm{Tr}(H\Sigma_n)=\frac1{2n}\mathrm{Tr}(HH^{-1}CH^{-1})=\frac1{2n}\mathrm{Tr}(H^{-1}C)$.

**7.B.i.2.** Expand $\hat R_n$ around its minimizer, where its gradient
vanishes:
$\hat R_n(\theta^*)\approx\hat R_n(\hat\theta_n)+\tfrac12\delta^\top\hat H\delta$,
with $\hat H=\nabla^2\hat R_n\approx H$. Take expectations, using
$\mathbb E\hat R_n(\theta^*)=R(\theta^*)$ because $\theta^*$ does not depend
on the sample:
$$
\mathbb E\hat R_n(\hat\theta_n)\approx R(\theta^*)-\tfrac1{2n}\mathrm{Tr}(H^{-1}C).
$$
Subtracting this from 1 gives
$\mathbb E[R(\hat\theta_n)-\hat R_n(\hat\theta_n)]\approx\big(\tfrac1{2n}+\tfrac1{2n}\big)\mathrm{Tr}(H^{-1}C)=\frac1n\mathrm{Tr}(H^{-1}C)$.

**7.B.i.3.** The test loss of $\hat\theta_n$ is above $R(\theta^*)$ by
$\mathrm{Tr}(H^{-1}C)/2n$, and its training loss is *below* $R(\theta^*)$ by
the same amount, because $\hat\theta_n$ was fitted to that sample's noise. The
optimism spans both. A held-out set estimates $R(\hat\theta_n)$, so held-out
minus training loss estimates the optimism. Held-out loss cannot give the
excess risk without the unknown $R(\theta^*)$, although differences of excess
risk between two models are measurable.

**7.B.ii.1.** $\nabla\ell=-(y-\theta x)x$, so $H=\mathbb E x^2=1$. At
$\theta^*=\beta$, $\nabla\ell=-\varepsilon x$, so
$C=\mathbb E[\varepsilon^2x^2]=\mathbb E[\tau^2x^2\cdot x^2]=3\tau^2$, using
$\mathbb E x^4=3$. Hence:

- $\mathrm{Var}\,\hat\theta_n\approx H^{-1}CH^{-1}/n=3\tau^2/n$;
- excess risk $\approx\mathrm{Tr}(H^{-1}C)/2n=3\tau^2/2n$;
- optimism $\approx3\tau^2/n$.

**7.B.ii.2.** The homoscedastic formulas use $C=\tau^2H$, i.e.
$\mathrm{Tr}(H^{-1}C)=\tau^2p$ with $p=1$. They give variance $\tau^2/n$,
excess risk $\tau^2/2n$ and optimism $\tau^2/n$: all too small by a factor of
3. The noise is largest exactly where $x$ is large, which is where the data
have the most leverage on $\hat\theta$.

Numerical check (`checks/wb2b_check.py`): for a model misspecified in both
mean and noise ($y=\sin2x+(0.3+0.5|x|)\xi$ fitted linearly with an intercept),
$\mathrm{Tr}(H^{-1}C)=3.19$ while the naive $\sigma^2p=2.01$. At $n=200$,
over $2\times10^4$ replications, the excess risk is $0.00812$ against the
formula's $0.00799$, and the optimism $0.01608$ against $0.01597$, a ratio of
$1.98$. The same check for a misspecified logistic regression (a non-quadratic
loss, $n=400$) gives a ratio of $1.98$.

### 7.C. The Fisher identity and AIC

#### 7.C.i. When $C=H$

1. Let $\ell(\theta;x,y)=-\log p_\theta(y\mid x)$, with the data generated by $p_{\theta^*}$ (a well-specified model). Prove that $C=H$ at $\theta^*$ by differentiating $\int p_\theta(y\mid x)\,dy=1$ twice.
2. Conclude that the optimism is $p/n$ and the excess risk $p/2n$. Relate this to AIC $=-2\log L(\hat\theta)+2p$.
3. For squared loss $\ell=\tfrac12(y-f_\theta(x))^2$ with $y=f_{\theta^*}(x)+\varepsilon$, $\varepsilon$ independent of $x$ and of variance $\sigma^2$, show that $C=\sigma^2H$, so $\mathrm{Tr}(H^{-1}C)=\sigma^2p$. Check this against OLS from §1, minding the convention.

**1.** Write $s=\nabla_\theta\log p_\theta$ (the score). Differentiating
$\int p_\theta\,dy=1$ once gives
$0=\int\nabla p_\theta=\int p_\theta\,s$, so $\mathbb E_{p_\theta}s=0$.
Differentiating again gives
$0=\int\nabla^2p_\theta=\int p_\theta\big(\nabla^2\log p_\theta+ss^\top\big)$,
using $\nabla^2p/p=\nabla^2\log p+ss^\top$. So
$\mathbb E_{p_\theta}[ss^\top]=-\mathbb E_{p_\theta}[\nabla^2\log p_\theta]$.
At $\theta=\theta^*$ the data come from $p_{\theta^*}$, so averaging also over
$x$ gives $C=\mathbb E[ss^\top]=\mathbb E[\nabla^2\ell]=H$.

**2.** $\mathrm{Tr}(H^{-1}C)=\mathrm{Tr}(I_p)=p$. So the optimism is $p/n$ and
the excess risk, which here is the KL to the truth, is $p/2n$ nats per
sample. Estimating the expected test log loss by $\hat R_n(\hat\theta_n)+p/n$
and multiplying by $2n$ gives $-2\log L(\hat\theta)+2p$: AIC is TIC with
$C=H$ assumed. This is "parameter counting", and it is exact only for
well-specified regular models.

**3.** $\nabla\ell=-r\nabla f$ with $r=y-f_\theta(x)$, and
$\nabla^2\ell=\nabla f\nabla f^\top-r\nabla^2f$. At $\theta^*$, $r=\varepsilon$.
So $C=\mathbb E[\varepsilon^2\nabla f\nabla f^\top]=\sigma^2\mathbb E[\nabla f\nabla f^\top]$
by independence, and $H=\mathbb E[\nabla f\nabla f^\top]-\mathbb E[\varepsilon\nabla^2f]=\mathbb E[\nabla f\nabla f^\top]$
since $\mathbb E[\varepsilon\mid x]=0$. Hence $C=\sigma^2H$.

For OLS with $x\sim\mathcal N(0,I)$, this predicts excess risk $\sigma^2p/2n$
in the $\tfrac12$ convention, i.e. $\sigma^2p/n$ in §1's $(y-f)^2$
convention. That matches the large-$n$ limit of §1's exact
$\sigma^2p/(n-p-1)$.

### 7.D. The Bayesian side: Laplace and BIC

#### 7.D.i. The evidence to leading order

Take the log loss, a prior density $\pi$ that is continuous and positive at
$\theta^*$, and the marginal likelihood (evidence)
$$
Z_n=\int\pi(\theta)\,e^{-n\hat R_n(\theta)}\,d\theta .
$$

1. Use the Laplace approximation around $\hat\theta_n$ to show that
$$
-\log Z_n=n\hat R_n(\hat\theta_n)+\frac p2\log n+\tfrac12\log\det\hat H-\frac p2\log2\pi-\log\pi(\hat\theta_n)+o(1).
$$
Hence $-2\log Z_n=-2\log L(\hat\theta_n)+p\log n+O(1)$: BIC.
2. AIC's penalty is $2p$ and BIC's is $p\log n$. They answer different questions. Which question does each answer?

**1.** Near $\hat\theta_n$,
$n\hat R_n(\theta)\approx n\hat R_n(\hat\theta_n)+\frac n2(\theta-\hat\theta_n)^\top\hat H(\theta-\hat\theta_n)$,
and $\pi(\theta)\approx\pi(\hat\theta_n)$ over the region of width
$O(n^{-1/2})$ that dominates the integral. The Gaussian integral gives
$$
Z_n\approx\pi(\hat\theta_n)\,e^{-n\hat R_n(\hat\theta_n)}\,(2\pi)^{p/2}\det(n\hat H)^{-1/2},
$$
and $\det(n\hat H)=n^p\det\hat H$. Taking $-\log$ gives the stated
expansion. Every term except the first two is $O(1)$, so
$-2\log Z_n=-2\log L(\hat\theta_n)+p\log n+O(1)$.

**2.** AIC (and TIC) estimate the expected test loss of the plug-in
predictor $p_{\hat\theta_n}$: a prediction question. BIC approximates
$-2\log$ of the evidence, so differences of BIC approximate log posterior
odds between models: a model-identification question. The evidence penalizes
each parameter by $\tfrac12\log n$ because the posterior volume shrinks like
$n^{-p/2}$ relative to the prior (the Occam factor; WB4 §2). In singular
models this volume scaling changes: $\frac p2\log n$ becomes $\lambda\log n$
with $\lambda$ the real log canonical threshold, the central quantity of singular learning theory (SLT).

### 7.E. Three estimands (optional)

Use the log loss, a model $p_\theta(y\mid x)$, and a truth $q(y\mid x)$; take
the model to be well specified, $q=p_{\theta^*}$. Let
$\pi_n(\theta)\propto\pi(\theta)\prod_ip_\theta(y_i\mid x_i)$ be the
posterior, and $\bar p_n(y\mid x)=\int p_\theta(y\mid x)\,\pi_n(d\theta)$ the
posterior predictive. With $\mathbb E_D$ over the training sample:

1. **Single-network excess risk:** $\ \mathcal E=\mathbb E_D\,\mathbb E_x\,\mathrm{KL}\big(q\,\|\,p_{\hat\theta_n}\big)=\mathbb E_D R(\hat\theta_n)-R(\theta^*)$. One deterministic output of training.
2. **Bayes generalization error:** $\ B_g=\mathbb E_D\,\mathbb E_x\,\mathrm{KL}\big(q\,\|\,\bar p_n\big)$. The posterior-predictive ensemble.
3. **Gibbs generalization error:** $\ G_g=\mathbb E_D\,\mathbb E_{\theta\sim\pi_n}\,\mathbb E_x\,\mathrm{KL}\big(q\,\|\,p_\theta\big)$. One random draw from the posterior.

Each is an expected population loss minus the entropy $R(\theta^*)$ of the
truth, so all three are in nats per sample. (Watanabe, *Mathematical Theory
of Bayesian Statistics*, 2018, ch. 1–2, defines them without the outer
$\mathbb E_D$, states results for their expectations, and treats the singular
case. For the singular Bayes generalization error and WAIC, see also
[Watanabe (2010)](https://jmlr.org/papers/v11/watanabe10a.html).)

**Watanabe's regular-case results.** For a regular, well-specified model at
temperature 1, $\mathbb E\,B_g=\frac p{2n}+o(\frac1n)$ and
$\mathbb E\,G_g=\frac pn+o(\frac1n)$. Under the realizable analytic assumptions of singular learning theory,
$\mathbb E\,B_g=\frac\lambda n+o(\frac1n)$: the real log canonical threshold
$\lambda\le p/2$ replaces $p/2$.

#### 7.E.i. Deriving the regular case

1. Show that $B_g\le G_g$ always.
2. In the regular case, approximate the posterior by $\mathcal N\big(\hat\theta_n,(nH)^{-1}\big)$. Show that $G_g\approx p/n$.
3. Show that $B_g\approx p/2n$, the same as $\mathcal E$ to leading order.
4. Without well-specification, define $K^*=\mathbb E_x\mathrm{KL}(q\|p_{\theta^*})$. Keep the same Laplace posterior and compute $G_g-K^*$ in terms of $\mathrm{Tr}(H^{-1}C)$. What does this say about the posterior's width?

<details><summary>Hint</summary>

For 1, $\mathrm{KL}(q\|\cdot)$ is convex in its second argument. For 2, write a posterior draw as $\theta=\hat\theta_n+u$, with $\mathbb E[u\mid D]=0$ and $\delta=\hat\theta_n-\theta^*$ fixed given the data. For 3, expand $\log\mathbb E_u\,p_{\hat\theta_n+u}(y\mid x)$ to second order in $u$, and use the Fisher identity of 7.C.
</details>

**1.** $\bar p_n=\mathbb E_{\theta\sim\pi_n}p_\theta$, and $\mathrm{KL}(q\|\cdot)$
is convex, so by Jensen
$\mathrm{KL}(q\|\bar p_n)\le\mathbb E_{\theta\sim\pi_n}\mathrm{KL}(q\|p_\theta)$.
Average over $x$ and $D$.

**2.** For a draw $\theta=\theta^*+\delta+u$, the excess risk is
$\approx\tfrac12(\delta+u)^\top H(\delta+u)$. Given the data, $u$ has mean
zero, so the cross term vanishes in expectation, and
$$
G_g\approx\tfrac12\mathrm{Tr}\Big(H\big(\mathbb E\delta\delta^\top+\mathbb Euu^\top\big)\Big)
=\tfrac12\mathrm{Tr}\big(H(H^{-1}/n+H^{-1}/n)\big)=\frac pn,
$$
using $\mathbb E\delta\delta^\top=H^{-1}CH^{-1}/n=H^{-1}/n$ (since $C=H$).
The draw pays the ERM's error once and the posterior's own spread once more.

**3.** Write $\ell=-\log p_\theta$, $g=\nabla\ell(\hat\theta_n)$ and
$h=\nabla^2\ell(\hat\theta_n)$ at a given $(x,y)$. Then
$$
\log\bar p_n=\log\mathbb E_u\,e^{-\ell(\hat\theta_n+u)}
\approx-\ell(\hat\theta_n)+\log\mathbb E_u\,e^{-u^\top g-\frac12u^\top hu}
\approx-\ell(\hat\theta_n)+\tfrac12\mathbb E_u\big[u^\top(gg^\top-h)u\big],
$$
to order $1/n$, since $\mathbb E u=0$. With $\mathbb Euu^\top=(nH)^{-1}$, the
correction is $\frac1{2n}\mathrm{Tr}\big(H^{-1}(gg^\top-h)\big)$. Averaging over
$(x,y)\sim q$ near $\theta^*$, $\mathbb E[gg^\top-h]=C-H=0$ by the Fisher
identity. So $-\mathbb E\log\bar p_n=R(\hat\theta_n)+o(1/n)$, and
$B_g\approx\mathcal E\approx p/2n$.

In the regular case the ensemble buys nothing at leading order. The three
estimands differ by constant factors ($\tfrac12,\tfrac12,1$ times $p/n$). In
singular models they come apart: $B_g\approx\lambda/n$, while these regular-case arguments do not determine the MLE's leading error.

Numerical check (`checks/wb2b_check.py`), with linear-Gaussian regression,
$p=5$, $n=100$, known noise and a flat prior, so that the posterior is exactly
Gaussian: $\mathcal E=0.0268$ (exact finite-$n$ value
$p/2(n-p-1)=0.0266$), $B_g=0.0258$, $G_g=0.0526$, against $p/2n=0.025$ and
$p/n=0.05$.

**4.** Now $\mathbb E\delta\delta^\top=H^{-1}CH^{-1}/n$ while the posterior
still has covariance $H^{-1}/n$, so
$G_g-K^*\approx\frac1{2n}\big(\mathrm{Tr}(H^{-1}C)+p\big)$.
The approximation error $K^*$ need not vanish under misspecification. The posterior's width
is set by $H$ alone, but the actual sampling spread of $\hat\theta_n$ is set by
the sandwich. A misspecified posterior is overconfident when $C\succ H$ and
underconfident when $C\prec H$. This is one honest motivation for tempering.

#### 7.E.ii. Which estimand do practical tools measure?

For each of the following, say which of the three estimands it measures or
controls, and in which regime the link is a theorem: held-out loss, early
stopping, ridge (weight decay), and flatness $\tfrac12\mathrm{Tr}(H\Sigma)$.

- **Held-out loss** estimates the population loss of whatever predictor you
  evaluate: one trained network (estimand 1, up to the unknown constant
  $R(\theta^*)$), an ensemble (closer to 2), or a posterior sample (3). It is
  model-free and needs no regime assumption, which is why it is the
  practical standard.
- **Early stopping** and **ridge** change the estimator; they do not choose
  an estimand. They act on the single-network excess risk by trading bias
  for variance (§3–§4). Ridge is MAP under a Gaussian prior, which is neither
  the Bayes ensemble nor a Gibbs draw. (In linear-Gaussian models the
  posterior mean equals the MAP, so there it coincides with the ensemble's
  mean prediction.)
- **Flatness** $\tfrac12\mathrm{Tr}(H\Sigma)$ approximates the mean local
  excess risk when $\Sigma=\mathbb E[(\theta-\theta^*)(\theta-\theta^*)^\top]$.
  This is a second moment, equal to the covariance only for a centered parameter. If $\Sigma$ is the sampling covariance of
  $\hat\theta_n$, it is estimand 1 (7.B). If $\Sigma$ is the posterior's
  covariance, it is the posterior's contribution to estimand 3. If $\Sigma$ is
  SGD's stationary covariance, it is a Gibbs-like error for the "SGD
  posterior" (§8). Which one is meant has to be stated.
- None of the regular-case formulas is a theorem for overparametrized
  networks trained by SGD: $H$ can be singular, and the empirical gradient
  covariance can vanish at squared-loss interpolation even when the
  population covariance does not (§8).

<a id="wb2-section-8"></a>

## 8. Does it cancel? $C$ vs $H$

**The result.** The clean formulas of §7 lean on $C\propto H$, which holds for
a well-specified likelihood at the optimum. Per-sample, both matrices are
built from the same rank-one pieces, but weighted differently, and at
squared-loss interpolation the empirical gradient covariance vanishes
while empirical curvature can remain nonzero. Empirically the diagonals follow
$C_{ii}\propto H_{ii}^\gamma$, with fitted $\gamma$ roughly between 1 and 1.5,
and a moment argument (8.B) makes $[1,2]$ the natural range, though it does
not force a fitted exponent into it. On a quadratic basin, in
the continuous-time approximation, SGD's stationary covariance solves
$H\Sigma+\Sigma H=(\eta/B)C$. It matches a
tempered posterior $\frac TnH^{-1}$ if and only if $C\propto I$, i.e.
$\gamma=0$, which is not the measured range. In the continuous-time approximation its quadratic excess risk is
$\eta\,\mathrm{Tr}\,C/4B$ exactly; discrete SGD has a step-size correction.

**Setup, self-contained.** Loss $\ell(\theta;z)$, population risk $R$,
minimizer $\theta^*$ (shifted to $0$ where convenient),
$H=\nabla^2R(\theta^*)$, and gradient-noise covariance
$C=\mathrm{Cov}[\nabla\ell(\theta;z)]$. From §4: a parameter
$\theta^*+\delta$ with $\mathbb E\,\delta\delta^\top=\Sigma$ has excess risk
$\approx\tfrac12\mathrm{Tr}(H\Sigma)$. From §7: for a well-specified log loss,
$C=H$ at $\theta^*$, and for squared loss with independent noise, $C=\sigma^2H$.
A *tempered posterior* is $\pi_T(\theta)\propto\pi(\theta)\,e^{-n\hat R_n(\theta)/T}$;
near a nondegenerate mode it has covariance $\approx\frac TnH^{-1}$.

### 8.A. Per-sample structure

*Source:* Zhang, Yang and Tu (reading in 8.D), §2.1, on why the Fisher
approximation $C\approx H$ fails.

#### 8.A.i. Same pieces, different weights

1. *Squared loss.* Let $\ell_i=\tfrac12r_i^2$ with residual $r_i=f_\theta(x_i)-y_i$, and $v_i=\nabla f_\theta(x_i)$. Write the per-sample gradient $g_i$ and the per-sample Gauss–Newton matrix $h_i$ (the Hessian without the $r_i\nabla^2f$ term). Show that $h_i$ has rank at most one, and that over the training set
$$
H_{GN}=\frac1n\sum_ih_i,\qquad C=\frac1n\sum_ir_i^2h_i-\bar g\bar g^\top .
$$
2. At an interpolating minimum ($r_i=0$ for all $i$), what are $C$ and $H_{GN}$? Away from interpolation, under what condition is $C=\sigma^2H_{GN}$?
3. *Logistic loss.* With $y_i\in\{0,1\}$, $s_i=\sigma(f_\theta(x_i))$ and $\ell_i=-y_i\log s_i-(1-y_i)\log(1-s_i)$, show that $g_i=(s_i-y_i)v_i$ and that the per-sample Gauss–Newton matrix is $h_i=a_iv_iv_i^\top$ with $a_i=s_i(1-s_i)$. For a correctly classified example with large margin, compare the weights of $v_iv_i^\top$ in $g_ig_i^\top$ and in $h_i$. Which goes to zero faster as the margin grows?

**1.** $g_i=\nabla\ell_i=r_iv_i$ and $\nabla^2\ell_i=v_iv_i^\top+r_i\nabla^2f$,
so $h_i=v_iv_i^\top$, which has rank at most one (rank one when $v_i\ne0$). Averaging gives $H_{GN}$. The
empirical gradient covariance is
$\frac1n\sum_ig_ig_i^\top-\bar g\bar g^\top=\frac1n\sum_ir_i^2v_iv_i^\top-\bar g\bar g^\top$.

**2.** At interpolation every $r_i=0$, so every $g_i=0$ and $C=0$. But
$H_{GN}=\frac1n\sum v_iv_i^\top\neq0$. So $C\propto H$ fails completely:
gradient noise vanishes in every direction while curvature does not.
Away from interpolation, at the population optimum with $r=-\varepsilon$
independent of $x$ (hence of $v$) and $\mathbb E\varepsilon=0$:
$\bar g=0$ and $C=\mathbb E[\varepsilon^2]\,\mathbb E[vv^\top]=\sigma^2H_{GN}$.
The cancellation needs noise that is independent of where you are in input
space, i.e. a well-specified model with homoscedastic noise.

**3.** $\partial\ell_i/\partial f=s_i-y_i$, so $g_i=(s_i-y_i)v_i$, and
$\partial^2\ell_i/\partial f^2=s_i(1-s_i)$, giving $h_i=a_iv_iv_i^\top$. For
$y_i=1$ with margin $f\to\infty$: $g_ig_i^\top$ has weight
$(1-s_i)^2\approx e^{-2f}$, and $h_i$ has weight $a_i\approx e^{-f}$. So
$g_ig_i^\top\approx a_i^2v_iv_i^\top$: the noise weight is the *square* of
the curvature weight. Both vanish as the network separates the data, but the
noise vanishes faster, so $C/H\to0$ again. (If $y$ were resampled from the
model, $\mathbb E_y(s-y)^2=s(1-s)=a$ and $C=H$: the Fisher identity. The
mismatch comes from using the observed labels, which the model fits.)

### 8.B. The diagonal power law

Zhang, Yang and Tu (2026) argue that, beyond the Fisher-equals-Hessian
setting, $C\propto\mathbb E_i[h_i^2]$ (with $h_i$ the per-sample Hessian), and also report approximate commutation of
$C$ and $H=\mathbb E_i[h_i]$ in their setting. Commutation is an additional
property, not a general consequence of the second-moment relation. In the
fully connected layers they analyse, they fit per-layer power laws
$C_{ii}\propto H_{ii}^\gamma$, with fitted $\gamma$ roughly $1.1$–$1.5$ for
cross-entropy and close to $1$ for squared loss. Their §5 derives a pair of
moment bounds and reads them as confining $\gamma$ to $[1,2]$. This exercise
derives the same kind of bounds in a simplified model and asks what they do
and do not imply.

#### 8.B.i. Why between 1 and 2?

Take rank-one per-sample Hessians $h_i=v_iv_i^\top$ as in 8.A.i.1, and the
model $C:=\mathbb E_i[h_i^2]$ (the constant of proportionality does not affect
$\gamma$).

1. Show that $C_{jj}=\mathbb E_i\big[\|v_i\|^2v_{ij}^2\big]$ and $H_{jj}=\mathbb E_i[v_{ij}^2]$.
2. Prove the envelope $H_{jj}^2\le C_{jj}\le G^2H_{jj}$, where $G^2=\max_i\|v_i\|^2$, and note that $H_{jj}\le G^2$. Draw it on log–log axes.
3. Suppose each example's gradient lies along a single coordinate: $v_i=s_{k}e_{k}$ for examples of "type" $k$, which occur with frequency $\pi_k$. Compute $H_{kk}$ and $C_{kk}$. Show that $\gamma=1$ if curvature varies across coordinates only through the frequencies $\pi_k$, and $\gamma=2$ if only through the magnitudes $s_k$.
4. Does the envelope force a fitted $\gamma$ into $[1,2]$?

<details><summary>Hint</summary>

For the lower bound in 2, use $\|v_i\|^2\ge v_{ij}^2$, then Jensen (or Cauchy–Schwarz) on $\mathbb E\,v_{ij}^4$. For 3, each example contributes to exactly one coordinate.
</details>

**1.** $h_i^2=v_iv_i^\top v_iv_i^\top=\|v_i\|^2v_iv_i^\top$, whose $(j,j)$
entry is $\|v_i\|^2v_{ij}^2$. Average over $i$. Similarly $(h_i)_{jj}=v_{ij}^2$.

**2.** Upper bound: $\|v_i\|^2\le G^2$, so
$C_{jj}\le G^2\,\mathbb E v_{ij}^2=G^2H_{jj}$. Lower bound:
$\|v_i\|^2\ge v_{ij}^2$, so $C_{jj}\ge\mathbb E v_{ij}^4\ge(\mathbb Ev_{ij}^2)^2=H_{jj}^2$
by Cauchy–Schwarz. Also $H_{jj}\le\mathbb E\|v_i\|^2\le G^2$. On log–log axes
the points lie in a wedge between a line of slope 2 (below) and a line of
slope 1 (above), and the two lines meet at $H_{jj}=G^2$.

**3.** $H_{kk}=\pi_ks_k^2$ and $C_{kk}=\pi_ks_k^4$. If $s_k\equiv s$ and only
$\pi_k$ varies, $C_{kk}=s^2H_{kk}$: $\gamma=1$. If $\pi_k\equiv\pi$ and only
$s_k$ varies, $C_{kk}=H_{kk}^2/\pi$: $\gamma=2$. Curvature that comes from
*how often* a parameter is used gives $\gamma=1$; curvature that comes from
*how strongly* it is used on the examples that use it gives $\gamma=2$.
(Note that "all $v_i$ parallel" gives $\gamma=1$, not $2$: scaling every
gradient by a per-example factor changes all coordinates proportionally.)

**4.** No. A cloud that sits on the slope-1 line at small $H$ and on the
slope-2 line near $H=G^2$ can be fitted with a slope below 1, and the reverse
gives a slope above 2. The envelope makes $[1,2]$ natural, and 3 shows both
ends are attained by simple mechanisms. The measured range is an empirical
fact about which mechanism dominates. The checks are in `checks/wb2b_check.py`.

### 8.C. SGD's stationary law on a quadratic basin

Near the minimum (shifted to $0$), take $R(\theta)\approx\tfrac12\theta^\top H\theta$
with $H\succ0$, and SGD with learning rate $\eta$ and batch size $B$:
$$
\theta_{k+1}=\theta_k-\eta\,(H\theta_k+\xi_k),\qquad\mathbb E\,\xi_k=0,\quad\mathrm{Cov}\,\xi_k=C/B,
$$
with $\xi_k$ independent across steps and independent of $\theta_k$, and
$C$ fixed (state-independent near the minimum). For discrete-time stationarity
assume $0<\eta<2/\lambda_{\max}(H)$.

*Source:* this stationary-distribution analysis, and the idea of tuning
constant-step SGD so that its stationary law approximates a posterior, are in
[Mandt, Hoffman and Blei (2017)](https://arxiv.org/abs/1704.04289).

#### 8.C.i. The Lyapunov equation and the posterior

1. Show that the stationary covariance of the recursion satisfies $\Sigma_\eta=(I-\eta H)\Sigma_\eta(I-\eta H)+\eta^2C/B$, and that to leading order in $\eta$ this is the Lyapunov equation
$$
H\Sigma+\Sigma H=\frac\eta B\,C .
$$
(It is also the stationary equation of the Ornstein–Uhlenbeck process $d\theta=-H\theta\,dt+\sqrt{\eta/B}\,C^{1/2}dW$, with $t=\eta k$.)
2. Show that the Lyapunov equation has a unique solution for $H\succ0$. If $C$ and $H$ commute, show that $\Sigma=\frac\eta{2B}CH^{-1}$. With $C=cH^\gamma$, this is $\Sigma=\frac{\eta c}{2B}H^{\gamma-1}$.
3. Show that $\Sigma=\frac TnH^{-1}$ for some temperature $T$ if and only if $C\propto I$. Do not assume that $C$ and $H$ commute. Find $T$.
4. Show that $\tfrac12\mathrm{Tr}(H\Sigma)=\frac{\eta\,\mathrm{Tr}\,C}{4B}$ exactly, for any $C$.
5. With $C=cH^\gamma$, write three excess risks: ERM's $\frac1{2n}\mathrm{Tr}(H^{-1}C)$ from §7, SGD's stationary value, and the tempered posterior's $\tfrac12\mathrm{Tr}(H\cdot\frac TnH^{-1})$. What does each count?
6. *Preconditioning.* Replace the update by $\theta\leftarrow\theta-\eta P(H\theta+\xi)$ with $P\succ0$. Write the new Lyapunov equation. Assuming $C\succ0$, show that $P=C^{-1}$ gives $\Sigma=\frac\eta{2B}H^{-1}$.
7. For $\gamma\in[1,2]$, compare the shapes of $\Sigma$ and the posterior: along which directions does SGD spread more than the posterior? What would "SGD is approximately Bayesian" have to mean?

<details><summary>Hint</summary>

For 2, $\Sigma=\int_0^\infty e^{-Ht}\,\frac\eta BC\,e^{-Ht}dt$ solves the equation; for uniqueness, work in $H$'s eigenbasis, where the equation reads $(h_j+h_k)\Sigma_{jk}=\frac\eta BC_{jk}$. For 3, substitute $\Sigma=\kappa H^{-1}$. For 4, take the trace.
</details>

**1.** $\theta_k$ is independent of $\xi_k$, so
$\mathrm{Cov}\,\theta_{k+1}=(I-\eta H)\,\mathrm{Cov}\,\theta_k\,(I-\eta H)+\eta^2C/B$.
At stationarity, expanding gives
$\Sigma_\eta=\Sigma_\eta-\eta(H\Sigma_\eta+\Sigma_\eta H)+\eta^2H\Sigma_\eta H+\eta^2C/B$,
i.e. $H\Sigma_\eta+\Sigma_\eta H=\frac\eta BC+\eta H\Sigma_\eta H$. Since
$\Sigma_\eta=O(\eta)$, the last term is $O(\eta^2)$; dropping it gives the
Lyapunov equation. (The discrete law is wider in sharp directions, by a
relative $O(\eta h)$, and diverges at $\eta h\ge2$.)

**2.** In the eigenbasis of $H$ (eigenvalues $h_j>0$), the equation reads
$(h_j+h_k)\Sigma_{jk}=\frac\eta BC_{jk}$, which has the unique solution
$\Sigma_{jk}=\frac\eta B\,C_{jk}/(h_j+h_k)$. If $C$ commutes with $H$, $C$ is
diagonal in the same basis, so $\Sigma_{jj}=\frac\eta{2B}C_{jj}/h_j$, i.e.
$\Sigma=\frac\eta{2B}CH^{-1}$. With $C=cH^\gamma$,
$\Sigma=\frac{\eta c}{2B}H^{\gamma-1}$.

**3.** If $\Sigma=\kappa H^{-1}$, then $H\Sigma+\Sigma H=2\kappa I$, so
$\frac\eta BC=2\kappa I$ and $C\propto I$. Conversely, if $C=cI$ then
$\Sigma=\frac{\eta c}{2B}H^{-1}$ solves the equation, and by uniqueness it is
the solution. No commutation was assumed. Matching $\frac{\eta c}{2B}=\frac Tn$
gives $T=\frac{n\eta c}{2B}$.

**4.** Take the trace of the Lyapunov equation:
$\mathrm{Tr}(H\Sigma)+\mathrm{Tr}(\Sigma H)=2\,\mathrm{Tr}(H\Sigma)=\frac\eta B\mathrm{Tr}\,C$.
So $\tfrac12\mathrm{Tr}(H\Sigma)=\frac{\eta\,\mathrm{Tr}\,C}{4B}$, for any $C$,
commuting or not.

**5.** With $C=cH^\gamma$:

| | excess risk | counts |
|---|---|---|
| ERM (§7) | $\frac c{2n}\mathrm{Tr}\,H^{\gamma-1}$ | sampling noise in $\hat\theta_n$, each direction weighted by noise over curvature |
| SGD stationary | $\frac{\eta c}{4B}\mathrm{Tr}\,H^\gamma$ | total gradient noise per step; curvature does not enter separately |
| posterior | $\frac{Tp}{2n}$ | parameter count at temperature $T$; neither $H$ nor $C$ enters |

At $\gamma=1$ (the Fisher case), ERM gives $\frac{cp}{2n}$, parameter
counting, while SGD gives $\frac{\eta c}{4B}\mathrm{Tr}H$, which grows with
sharpness. For this $\gamma=1$ comparison at fixed $c$, the SGD excess-risk value increases with curvature. This local calculation does not by itself determine how SGD selects between basins. Its noise scale is
$\eta/B$, not $1/n$.

**6.** The update is $\theta\leftarrow(I-\eta PH)\theta-\eta P\xi$. The same
calculation gives $PH\Sigma+\Sigma HP=\frac\eta BPCP$. With $P=C^{-1}$,
try $\Sigma=\kappa H^{-1}$: the left side is $2\kappa C^{-1}$ and the right
side is $\frac\eta BC^{-1}$, so $\kappa=\frac\eta{2B}$. Preconditioning by the
inverse noise covariance makes the continuous-time stationary covariance
posterior-shaped when $C\succ0$. With Gaussian diffusion this also matches
the Gaussian stationary law; covariance matching alone does not establish
a Gaussian law for arbitrary discrete update noise. This is the idea behind using preconditioned SGD as an approximate
posterior sampler.

**7.** With $\gamma\in[1,2]$, $\Sigma\propto H^{\gamma-1}$ ranges from
isotropic ($\gamma=1$) to $\propto H$ ($\gamma=2$). The posterior
$\propto H^{-1}$ is long in flat directions and narrow in sharp ones. SGD's
cloud is *at least as wide in sharp directions as in flat ones*: the
opposite ordering. "SGD samples a (tempered) posterior" is the claim that
$\gamma\approx0$, i.e. isotropic gradient noise, or that one has
preconditioned by $C^{-1}$. Neither holds for plain SGD in the measured
regime.

#### 8.C.ii. The best temperature, and a symmetry (optional)

1. Show that $T^*=n\,\mathrm{Tr}(H\Sigma)/p$ minimizes $\mathrm{KL}\big(\mathcal N(0,\Sigma)\,\|\,\mathcal N(0,\frac TnH^{-1})\big)$ over $T$, that the minimum is
$$
\mathrm{KL}^*=\tfrac12\big[p\log\big(\mathrm{Tr}(H\Sigma)/p\big)-\log\det(H\Sigma)\big]\ \ge0,
$$
with equality iff $H\Sigma\propto I$, and that at $T=T^*$ the posterior's excess risk $\frac{T^*p}{2n}$ equals SGD's $\tfrac12\mathrm{Tr}(H\Sigma)$.
2. In the commuting case with $p=2$ and $H=\mathrm{diag}(1,\kappa)$, show that $\mathrm{KL}^*=\log\cosh\big(\tfrac\gamma2\log\kappa\big)$, which is symmetric in $\gamma\mapsto-\gamma$. Is $\mathrm{KL}^*$ symmetric in $\gamma$ for $p=3$?

<details><summary>Hint</summary>

For zero-mean Gaussians, $\mathrm{KL}(\mathcal N(0,\Sigma)\,\|\,\mathcal N(0,S))=\tfrac12\big[\mathrm{Tr}(S^{-1}\Sigma)-p+\log\det S-\log\det\Sigma\big]$. Minimize over $\log T$, then write the result in terms of the eigenvalues of $H\Sigma$ and compare their arithmetic and geometric means. For 2, in the commuting case $H\Sigma\propto H^\gamma$.
</details>

**1.** For Gaussians with mean zero,
$\mathrm{KL}(\mathcal N(0,\Sigma)\|\mathcal N(0,S))=\tfrac12[\mathrm{Tr}(S^{-1}\Sigma)-p+\log\det S-\log\det\Sigma]$.
With $S=\frac TnH^{-1}$:
$$
\mathrm{KL}(T)=\tfrac12\Big[\tfrac nT\mathrm{Tr}(H\Sigma)-p+p\log\tfrac Tn-\log\det(H\Sigma)\Big].
$$
$\partial_T\mathrm{KL}=\tfrac12[-\tfrac n{T^2}\mathrm{Tr}(H\Sigma)+\tfrac pT]=0$
gives $T^*=n\,\mathrm{Tr}(H\Sigma)/p$; the function is convex in $\log T$, so
this is the minimum. Substituting gives the stated $\mathrm{KL}^*$. With
$\mu_j$ the eigenvalues of $H\Sigma$ (positive, since $H\Sigma$ is similar to
$H^{1/2}\Sigma H^{1/2}$), $\mathrm{KL}^*=\frac p2[\log\bar\mu_{\rm arith}-\log\bar\mu_{\rm geom}]\ge0$
by AM–GM, with equality iff all $\mu_j$ are equal, i.e. $H\Sigma\propto I$.
Finally, $\frac{T^*p}{2n}=\tfrac12\mathrm{Tr}(H\Sigma)$. The KL-optimal
temperature is also the one that matches excess risks.

**2.** Commuting case: $H\Sigma\propto H^\gamma$ has eigenvalues
$\propto(1,\kappa^\gamma)$. With $x=1$, $y=\kappa^\gamma$,
$\mathrm{KL}^*=\log\frac{(x+y)/2}{\sqrt{xy}}=\log\cosh\big(\tfrac12\log\frac yx\big)=\log\cosh\big(\tfrac\gamma2\log\kappa\big)$,
which is even in $\gamma$. For $p=3$ it is not symmetric in general: the
arithmetic mean of $\mu^\gamma$ and of $\mu^{-\gamma}$, each relative to its
geometric mean, differ unless the log-spectrum is symmetric. For $H$ with
eigenvalues $(1,2,30)$, $\mathrm{KL}^*(1.5)=2.97$ while $\mathrm{KL}^*(-1.5)=1.88$
(`checks/wb2b_check.py`). The interactive figure after 8.C.iii uses $p=2$, so the
symmetry of its $\mathrm{KL}^*$ readout is a feature of $p=2$.

#### 8.C.iii. A two-dimensional example

Take $H=\mathrm{diag}(1,10)$: $\theta_1$ is flat and $\theta_2$ sharp. With
$C=cH^\gamma$, use 8.C.i.2 to find the axes and aspect ratio of SGD's
$\Sigma$ for $\gamma=0,1,2$, and compare each with the posterior's
$\propto H^{-1}$. Which way does each SGD ellipse point? What happens if $C$'s
eigenbasis is rotated relative to $H$'s? The interactive figure below draws
these ellipses.

- $\gamma=0$: $\Sigma\propto H^{-1}$, elongated along the flat $\theta_1$ with
  aspect ratio $\sqrt{10}$ in standard deviations. Same shape as the
  posterior.
- $\gamma=1$: $\Sigma=\frac{\eta c}{2B}I$, a circle.
- $\gamma=2$: $\Sigma\propto H$, elongated along the *sharp* $\theta_2$:
  rotated $90°$ from the posterior.
- Rotating $C$'s eigenbasis tilts $\Sigma$ off the axes. Since
  $\Sigma\propto H^{-1}$ requires $C\propto I$ (8.C.i.3), no temperature
  fixes the shape, and $\mathrm{KL}^*>0$ for every $T$.
- The discrete-time ellipse is slightly wider than the continuous one along
  $\theta_2$, by a relative $O(\eta\kappa)$.

This is as far as WB2 goes. The Training Dynamics materials take up
state-dependent noise, edge-of-stability effects and what happens away from a
quadratic basin.

▶ **Interactive figure:** [`sgd_stationary_vs_posterior.html`](assets/interactive/sgd_stationary_vs_posterior.html). Slide γ from 0 to 2 and watch the SGD ellipse rotate from the posterior's shape through a circle to the opposite orientation. Then rotate the noise (φ) and check that no temperature recovers the posterior shape. The readout panel shows ½Tr(HΣ) = ηTr C/4B, T*, and the discrete-time correction.

### 8.D. Readings and rabbit holes

**[On the superlinear relationship between SGD noise covariance and loss landscape curvature](https://arxiv.org/abs/2602.05600)**, Yikuan Zhang, Ning Yang, Yuhai Tu (arXiv, 2026) · *core*

The assumption that SGD noise covariance is proportional to the Hessian holds only under restrictive conditions that deep networks typically violate. Using an activity–weight duality, they find C ∝ E_p[h_p²] instead. They also report that C and H approximately commute without coinciding (a separate property; see 8.B), and that the diagonals follow per-layer power laws C_ii ∝ H_ii^γ, with fitted γ near 1–1.5 and an argument that 1 ≤ γ ≤ 2. Read §2 (why the Fisher-equals-Hessian argument fails: compare 8.A) and skim §3, whose §3.4 derives C ∝ E[h_p²]. Then read §4 (the measured power laws, on MLPs and small CNNs on MNIST and CIFAR-10) and §5 (the moment bounds on γ, which are the envelope of 8.B.i, and why cross-entropy gives larger γ than squared loss). Note which layers the fits cover before carrying γ into 8.C.

**[A circuit prior in NN-Bayes](https://www.lesswrong.com/posts/SqDHeuycNkERurtSc/a-circuit-prior-in-nn-bayes)**, Kaarel Hänni, Dmitry Vaintrob (LessWrong, 2026) · *rabbit hole*

Bayesian learning with neural networks has a circuit prior: when the target is computed by a small circuit, few samples suffice for good test accuracy, even for arbitrarily overparametrized networks, under particular prior scalings and caveats. A post with slides presenting a simplified version of the result, and open problems at the end. It is a statement about Bayesian learning (§7.E), not about SGD. The comment thread discusses exactly that gap (some circuit-learnable functions, like sparse parities, are hard for gradient descent). Read it with 8.C.i.7 in mind.

<a id="wb2-section-9"></a>

## 9. Fantastic generalization measures

This section is a reading, with one exercise that connects it to §4, §7 and
§8.

**[Fantastic generalization measures and where to find them](https://arxiv.org/abs/1912.02178)**, Yiding Jiang, Behnam Neyshabur, Hossein Mobahi, Dilip Krishnan, Samy Bengio (ICLR 2020) · *core*

A large controlled study: 2187 CIFAR-10 networks spanning 7 hyperparameters (over 10,000 networks with repeats and SVHN), all trained to the same training cross-entropy, 0.01, and more than 40 published complexity measures ranked by how well they predict the generalization gap. Many norm-based measures correlate negatively with the gap when the optimizer is stochastic, the product of layer spectral norms strongly so. Sharpness-based measures (PAC-Bayes bounds and Keskar et al.'s sharpness) do best overall, and measures of gradient noise and of optimization speed are also predictive. §2 defines the rank correlations, including a 'granulated' version that varies one hyperparameter at a time, and a conditional-independence test; §4 has the results; the appendix defines every measure.

### 9.A. Exercise: invariance, and the assumptions behind TIC

1. Let $\theta=A\phi$ with $A$ invertible be a reparametrization that leaves the network function fixed (or the linearization of a smooth one at the minimum). Show that at the minimum $H\to A^\top HA$ and $C\to A^\top CA$, and that $\mathrm{Tr}(H^{-1}C)$ is invariant. Which of the parameter $\ell_2$ norm, the gradient variance $\mathrm{Tr}\,C$, $\mathrm{Tr}\,H$ and the product of layer spectral norms are invariant under the layer rescaling of §4.E.i.3, and which under a general $A$?
2. Jiang et al. train every model to the same cross-entropy. Why? Using 8.A, what does this do to the per-sample gradients and to $\hat C$? Which assumptions behind the derivation of TIC in 7.B fail for these networks?
3. Jiang et al. report both the pooled Kendall $\tau$ and the granulated $\Psi$ (their §2.2). Why might a measure have a high pooled $\tau$ but a low $\Psi$?

<details><summary>Hint</summary>

For 1, per-sample gradients transform as $\nabla_\phi\ell=A^\top\nabla_\theta\ell$, and at a critical point the Hessian transforms as in 4.E.i.4. For 2, list the assumptions in §7's setup (fixed $p$ with $n\to\infty$, $H\succ0$, the trained parameter is the ERM) and check each.
</details>

**1.** Under a reparametrization $\theta=A\phi$ with $A$ invertible (or its
linearization, for a smooth one) that leaves the function fixed, gradients
transform as $\nabla_\phi=A^\top\nabla_\theta$, so $H\to A^\top HA$ and
$C\to A^\top CA$ at the minimum. Then
$\mathrm{Tr}(H^{-1}C)\to\mathrm{Tr}(A^{-1}H^{-1}A^{-\top}A^\top CA)=\mathrm{Tr}(H^{-1}C)$:
TIC is invariant. (More generally, the eigenvalues of $H^{-1}C$ are.)

- *Not invariant:* the parameter $\ell_2$ norm, the raw gradient variance
  $\mathrm{Tr}\,C$, and plain sharpness ($\mathrm{Tr}\,H$ or a worst-case
  perturbation of fixed size).
- *Invariant to the layer rescaling of §4 but not to general
  reparametrizations:* the product of layer spectral norms, since
  $\|\alpha W_1\|\,\|W_2/\alpha\|=\|W_1\|\,\|W_2\|$ in a ReLU network.
- *Functions of the function:* the margin, and the gap itself.

**2.** Fixing the training loss removes a trivial predictor: models with
higher training loss have a different gap for uninteresting reasons. The
price is that every model is near interpolation, where per-sample gradients
are small (8.A). $\hat C$ is then small and dominated by the hardest
examples, and the regular-case theory behind $\mathrm{Tr}(H^{-1}C)$ is
furthest from applying. The assumptions that fail:
- *Interpolation* (8.A): at exact squared-loss interpolation, the empirical
  $\hat C$ is zero even though population gradient variability and test
  loss can remain positive. Plug-in TIC then loses its regular-regime
  justification. Jiang et al. stop at a small positive cross-entropy,
  not exact interpolation; for cross-entropy both curvature and gradient
  covariance can shrink, at different rates.
- *Singular $H$*: overparametrized networks have flat directions, so
  $H^{-1}$ is undefined and damped versions depend on $\epsilon$. The
  regular-case $p/2$ becomes the RLCT (§7.D–7.E), which TIC does not see.
- *SGD is not ERM*: the trained parameter's spread is the stationary law of
  §8.C, set by $\eta/B$ and $\mathrm{Tr}\,C$, not the sandwich. Learning-rate
  and batch-size axes move the gap through a channel TIC does not model.
  ($\mathrm{Tr}\,C$ itself does model it, which is one reading of Jiang et
  al.'s gradient-variance result.)
- *Wrong loss*: TIC is about the training loss. The 0–1 gap can move while
  the cross-entropy optimism does not, and conversely.
- *Ensembles*: TIC is about a single fitted network. If what you report is
  an ensemble's error, the leading constant can change (§7.E).

Where the assumptions do roughly hold ($p$ small compared with $n$, a
nondegenerate minimum, training loss bounded away from zero, a trained
parameter close to the ERM), comparisons within one architecture across data
size or regularization strength are the natural use of TIC.

**3.** If one hyperparameter (say, width) moves both the measure and the gap
strongly, the pooled $\tau$ is high even if the measure tracks nothing within
any fixed setting of the others. $\Psi$ only counts comparisons where one
hyperparameter changes, so a measure that just tracks one axis scores well on
that axis and near zero on the others. A measure that predicts along one axis
and not another is tracking the axis, not the gap.